# Historical research notebook — not the canonical final Pure implementation

This notebook is retained for research provenance and training logic. Its original Pure post-processing branch predates the final corrected Pure-specific spatial implementation used for the reported manuscript results. It must not be used as the authoritative final Pure scoring path.

For the final reported Full/Pure scoring logic, use `src/proposed_method.py` and `scripts/reproduce_full_pure.py`.


In [ ]:
import os
# =========================
# 1. Imports, paths, and data loading
# =========================

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

SEED = 42

# Clean proposed-method artifacts
CLEAN_RESULTS_DIR = Path(
    "03_Proposed_Results_Seed_42"
)

# Store all injected datasets and outputs separately
INJECTED_RESULTS_DIR = Path(
    "03_Injected_Proposed_Results_Seed_42"
)

INJECTED_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert CLEAN_RESULTS_DIR.exists(), (
    f"Clean-results folder not found: "
    f"{CLEAN_RESULTS_DIR.resolve()}"
)

# Raw dataset paths
X_path = Path("X.npy")
y_path = Path("y.npy")

assert X_path.exists(), f"Missing file: {X_path.resolve()}"
assert y_path.exists(), f"Missing file: {y_path.resolve()}"

X = np.load(
    X_path,
    mmap_mode=None
)

y = np.load(
    y_path,
    mmap_mode=None
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X dtype:", X.dtype)
print("y dtype:", y.dtype)

# =========================
# Basic checks
# =========================

assert X.ndim == 2, (
    "X must be a two-dimensional array."
)

assert y.ndim in (1, 2), (
    "y must be one- or two-dimensional."
)

assert X.shape[0] == y.shape[0], (
    "X and y must contain the same number of samples."
)

expected_features = 23

assert X.shape[1] == expected_features, (
    f"Expected {expected_features} features, "
    f"but found {X.shape[1]}."
)

assert np.isfinite(X).all(), (
    "X contains NaN or infinite values."
)

print("\nBasic data checks passed.")
print("Clean artifacts:", CLEAN_RESULTS_DIR.resolve())
print("Injected outputs:", INJECTED_RESULTS_DIR.resolve())

In [ ]:
# =========================
# 2. Generate synthetic anomaly scenarios
# =========================

n_samples, n_features = X.shape

inject_ratio = 0.0005  # 0.05%

n_inject = int(
    np.floor(
        n_samples * inject_ratio
    )
)

if n_inject < 1:
    raise ValueError(
        "The selected injection ratio produces no injected samples."
    )

print("Total samples:", n_samples)
print("Injection ratio:", inject_ratio)
print("Injected anomalies:", n_inject)

# -------------------------
# Fixed injected indices
# -------------------------

rng = np.random.default_rng(
    SEED
)

inject_idx = rng.choice(
    n_samples,
    size=n_inject,
    replace=False
)

inject_idx = np.sort(
    inject_idx
).astype(
    np.int64,
    copy=False
)

np.save(
    INJECTED_RESULTS_DIR / "inject_indices.npy",
    inject_idx
)

# Binary ground-truth injection mask
injection_labels = np.zeros(
    n_samples,
    dtype=np.uint8
)

injection_labels[inject_idx] = 1

np.save(
    INJECTED_RESULTS_DIR / "injection_ground_truth_labels.npy",
    injection_labels
)

print("Saved injection indices and ground-truth labels.")

# -------------------------
# Raw feature indices
# -------------------------

feature_idx = {
    "elevation": 0,
    "slope": 1,
    "flow_acc": 2,
    "profile_curv": 3,
    "plan_curv": 4,
    "depression_depth": 5,
    "local_std": 6,
    "elev_percentile": 7,
    "elev_contrast": 8,
    "slope_variability": 9,
    "flow_contrast": 10,
    "road_dist": 11,
    "river_dist": 12,
    "ndvi": 13,
    "ndwi": 14,
    "ndbi": 15,
}

# Land-use columns 16:23 remain unchanged.
continuous_cols = np.arange(
    0,
    16
)

# =========================
# Scenario 1: Extreme perturbation
# =========================

X_extreme = X.astype(
    np.float32,
    copy=True
)

X_extreme[
    inject_idx,
    feature_idx["elevation"]
] *= 3.0

X_extreme[
    inject_idx,
    feature_idx["slope"]
] *= 4.0

X_extreme[
    inject_idx,
    feature_idx["flow_acc"]
] *= 5.0

X_extreme[
    inject_idx,
    feature_idx["flow_contrast"]
] *= 3.0

np.save(
    INJECTED_RESULTS_DIR / "X_injected_extreme.npy",
    X_extreme
)

del X_extreme

print("Saved: X_injected_extreme.npy")

# =========================
# Scenario 2A: Strong noise
# =========================

X_noise_strong = X.astype(
    np.float32,
    copy=True
)

strong_noise = rng.normal(
    loc=5.0,
    scale=2.0,
    size=(
        n_inject,
        len(continuous_cols)
    )
).astype(
    np.float32
)

X_noise_strong[
    np.ix_(
        inject_idx,
        continuous_cols
    )
] += strong_noise

np.save(
    INJECTED_RESULTS_DIR / "X_injected_noise_strong.npy",
    X_noise_strong
)

del X_noise_strong, strong_noise

print("Saved: X_injected_noise_strong.npy")

# =========================
# Scenario 2B: Moderate noise
# =========================

X_noise_moderate = X.astype(
    np.float32,
    copy=True
)

moderate_noise = rng.normal(
    loc=0.0,
    scale=0.5,
    size=(
        n_inject,
        len(continuous_cols)
    )
).astype(
    np.float32
)

X_noise_moderate[
    np.ix_(
        inject_idx,
        continuous_cols
    )
] += moderate_noise

np.save(
    INJECTED_RESULTS_DIR / "X_injected_noise_moderate.npy",
    X_noise_moderate
)

del X_noise_moderate, moderate_noise

print("Saved: X_injected_noise_moderate.npy")

# =========================
# Scenario 2C: Mild noise
# =========================

X_noise_mild = X.astype(
    np.float32,
    copy=True
)

mild_noise = rng.normal(
    loc=0.0,
    scale=0.2,
    size=(
        n_inject,
        len(continuous_cols)
    )
).astype(
    np.float32
)

X_noise_mild[
    np.ix_(
        inject_idx,
        continuous_cols
    )
] += mild_noise

np.save(
    INJECTED_RESULTS_DIR / "X_injected_noise_mild.npy",
    X_noise_mild
)

del X_noise_mild, mild_noise

print("Saved: X_injected_noise_mild.npy")

# =========================
# Scenario 3: Cross-feature inconsistency
# =========================

X_inconsistent = X.astype(
    np.float32,
    copy=True
)

# Use independently permuted donor rows.
donor_idx = rng.permutation(
    inject_idx
)

# Topographic features
topographic_cols = [
    feature_idx["elevation"],
    feature_idx["slope"],
    feature_idx["profile_curv"],
    feature_idx["plan_curv"],
    feature_idx["depression_depth"],
]

X_inconsistent[
    np.ix_(
        inject_idx,
        topographic_cols
    )
] = X[
    np.ix_(
        donor_idx,
        topographic_cols
    )
]

# Hydrological inconsistency
hydrological_cols = [
    feature_idx["flow_acc"],
    feature_idx["flow_contrast"],
]

X_inconsistent[
    np.ix_(
        inject_idx,
        hydrological_cols
    )
] = X[
    np.ix_(
        donor_idx,
        hydrological_cols
    )
]

# Distance inconsistency
distance_cols = [
    feature_idx["road_dist"],
    feature_idx["river_dist"],
]

X_inconsistent[
    np.ix_(
        inject_idx,
        distance_cols
    )
] = X[
    np.ix_(
        donor_idx,
        distance_cols
    )
]

# Spectral inconsistency:
# inject NDBI values into the NDVI channel.
X_inconsistent[
    inject_idx,
    feature_idx["ndvi"]
] = X[
    donor_idx,
    feature_idx["ndbi"]
]

np.save(
    INJECTED_RESULTS_DIR / "X_injected_inconsistent.npy",
    X_inconsistent
)

del X_inconsistent

print("Saved: X_injected_inconsistent.npy")

print("\nAll injected scenarios generated successfully.")

In [ ]:
# =========================
# 3. Select injected scenario
# =========================

SCENARIO = "extreme"

scenario_files = {
    "extreme": "X_injected_extreme.npy",
    "noise_strong": "X_injected_noise_strong.npy",
    "noise_moderate": "X_injected_noise_moderate.npy",
    "noise_mild": "X_injected_noise_mild.npy",
    "inconsistent": "X_injected_inconsistent.npy",
}

if SCENARIO not in scenario_files:
    raise ValueError(
        f"Unknown scenario '{SCENARIO}'. "
        f"Available scenarios: {list(scenario_files)}"
    )

scenario_path = (
    INJECTED_RESULTS_DIR
    / scenario_files[SCENARIO]
)

X_injected = np.load(
    scenario_path,
    mmap_mode=None
)

inject_idx = np.load(
    INJECTED_RESULTS_DIR
    / "inject_indices.npy"
)

injection_labels = np.load(
    INJECTED_RESULTS_DIR
    / "injection_ground_truth_labels.npy"
)

assert X_injected.shape == X.shape
assert inject_idx.shape == (n_inject,)
assert int(injection_labels.sum()) == n_inject
assert np.isfinite(X_injected).all()

print("Selected scenario:", SCENARIO)
print("Injected dataset:", scenario_path.name)
print("Injected shape:", X_injected.shape)
print("Injected samples:", len(inject_idx))

In [ ]:
# =========================
# 4. Define corrected feature groups
# =========================

feature_groups = {
    "terrain": [
        "elevation",
        "slope",
        "profile_curv",
        "plan_curv",
        "depression_depth",
    ],

    "hydrological": [
        "flow_acc",
        "flow_contrast",
        "river_dist",
    ],

    "spatial": [
        "local_std",
        "elev_percentile",
        "elev_contrast",
        "slope_variability",
    ],

    "surface": [
        "ndvi",
        "ndwi",
        "ndbi",
    ],

    "infrastructure": [
        "road_dist",
    ],

    "landuse": [
        "landuse_1",
        "landuse_2",
        "landuse_3",
        "landuse_4",
        "landuse_5",
        "landuse_6",
        "landuse_7",
    ],
}

for view_name, features in feature_groups.items():
    print(
        f"{view_name}: "
        f"{len(features)} features"
    )

In [ ]:
# =========================
# 5. Define raw feature names
# =========================

feature_names = [
    "elevation",            # 0
    "slope",                # 1
    "flow_acc",             # 2
    "profile_curv",         # 3
    "plan_curv",            # 4
    "depression_depth",     # 5
    "local_std",            # 6
    "elev_percentile",      # 7
    "elev_contrast",        # 8
    "slope_variability",    # 9
    "flow_contrast",        # 10
    "road_dist",            # 11
    "river_dist",           # 12
    "ndvi",                 # 13
    "ndwi",                 # 14
    "ndbi",                 # 15
    "landuse_1",            # 16
    "landuse_2",            # 17
    "landuse_3",            # 18
    "landuse_4",            # 19
    "landuse_5",            # 20
    "landuse_6",            # 21
    "landuse_7",            # 22
]

assert len(feature_names) == X.shape[1]

feature_name_to_index = {
    name: idx
    for idx, name in enumerate(feature_names)
}

print("Feature names defined:", len(feature_names))

In [ ]:
# =========================
# 6. Create corrected feature views
# =========================

terrain_idx = [
    0,   # elevation
    1,   # slope
    3,   # profile_curv
    4,   # plan_curv
    5,   # depression_depth
]

hydro_idx = [
    2,   # flow_acc
    10,  # flow_contrast
    12,  # river_dist
]

spatial_idx = [
    6,   # local_std
    7,   # elev_percentile
    8,   # elev_contrast
    9,   # slope_variability
]

surface_idx = [
    13,  # ndvi
    14,  # ndwi
    15,  # ndbi
]

infra_idx = [
    11,  # road_dist
]

landuse_idx = list(
    range(16, 23)
)

# Confirm disjoint and complete partition.
all_view_indices = (
    terrain_idx
    + hydro_idx
    + spatial_idx
    + surface_idx
    + infra_idx
    + landuse_idx
)

assert len(all_view_indices) == 23
assert len(set(all_view_indices)) == 23
assert sorted(all_view_indices) == list(range(23))

# -------------------------
# Original clean views
# -------------------------

X_terrain = X[:, terrain_idx]
X_hydro = X[:, hydro_idx]
X_spatial = X[:, spatial_idx]
X_surface = X[:, surface_idx]
X_infra = X[:, infra_idx]
X_landuse = X[:, landuse_idx]

# -------------------------
# Injected views
# -------------------------

X_terrain_injected = X_injected[:, terrain_idx]
X_hydro_injected = X_injected[:, hydro_idx]
X_spatial_injected = X_injected[:, spatial_idx]
X_surface_injected = X_injected[:, surface_idx]
X_infra_injected = X_injected[:, infra_idx]
X_landuse_injected = X_injected[:, landuse_idx]

print("Corrected clean and injected views created.")

In [ ]:
# =========================
# 7. Sanity-check clean and injected views
# =========================

clean_views = {
    "terrain": X_terrain,
    "hydrological": X_hydro,
    "spatial": X_spatial,
    "surface": X_surface,
    "infrastructure": X_infra,
    "landuse": X_landuse,
}

injected_views = {
    "terrain": X_terrain_injected,
    "hydrological": X_hydro_injected,
    "spatial": X_spatial_injected,
    "surface": X_surface_injected,
    "infrastructure": X_infra_injected,
    "landuse": X_landuse_injected,
}

expected_dimensions = {
    "terrain": 5,
    "hydrological": 3,
    "spatial": 4,
    "surface": 3,
    "infrastructure": 1,
    "landuse": 7,
}

for view_name in expected_dimensions:
    clean_view = clean_views[view_name]
    injected_view = injected_views[view_name]

    expected_shape = (
        n_samples,
        expected_dimensions[view_name]
    )

    assert clean_view.shape == expected_shape
    assert injected_view.shape == expected_shape

    assert np.isfinite(clean_view).all()
    assert np.isfinite(injected_view).all()

    print(
        f"{view_name:15s} "
        f"clean={clean_view.shape}, "
        f"injected={injected_view.shape}"
    )

print("\nAll view checks passed.")

In [ ]:
# =========================
# 8. Terrain preprocessing using clean-run parameters
# =========================

terrain_bounds_path = (
    CLEAN_RESULTS_DIR
    / "terrain_clip_bounds.pkl"
)

assert terrain_bounds_path.exists(), (
    f"Missing terrain bounds: "
    f"{terrain_bounds_path.resolve()}"
)

terrain_clip_bounds = joblib.load(
    terrain_bounds_path
)

print("Loaded terrain clipping bounds:")
print(terrain_clip_bounds)

required_bound_keys = {
    "profile_curv",
    "plan_curv",
}

assert required_bound_keys.issubset(
    terrain_clip_bounds.keys()
), (
    "terrain_clip_bounds.pkl does not contain "
    "the expected curvature feature keys."
)

# Internal terrain order:
# 0 elevation
# 1 slope
# 2 profile_curv
# 3 plan_curv
# 4 depression_depth

def preprocess_terrain_with_clean_bounds(
    terrain_array,
    clip_bounds
):
    processed = terrain_array.astype(
        np.float32,
        copy=True
    )

    curvature_positions = {
        "profile_curv": 2,
        "plan_curv": 3,
    }

    for feature_name, position in curvature_positions.items():
        lower = float(
            clip_bounds[feature_name]["lower"]
        )

        upper = float(
            clip_bounds[feature_name]["upper"]
        )

        processed[:, position] = np.tanh(
            np.clip(
                processed[:, position],
                lower,
                upper
            )
        )

    # Depression depth was transformed in the clean pipeline
    # using non-negative log1p.
    processed[:, 4] = np.log1p(
        np.clip(
            processed[:, 4],
            0.0,
            None
        )
    )

    if not np.isfinite(processed).all():
        raise ValueError(
            "Terrain preprocessing produced "
            "NaN or infinite values."
        )

    return processed


X_terrain_proc = preprocess_terrain_with_clean_bounds(
    X_terrain,
    terrain_clip_bounds
)

X_terrain_injected_proc = (
    preprocess_terrain_with_clean_bounds(
        X_terrain_injected,
        terrain_clip_bounds
    )
)

assert X_terrain_proc.shape == (
    n_samples,
    5
)

assert X_terrain_injected_proc.shape == (
    n_samples,
    5
)

print("Terrain preprocessing completed.")
print("Clean terrain:", X_terrain_proc.shape)
print(
    "Injected terrain:",
    X_terrain_injected_proc.shape
)

In [ ]:
# =========================
# 9. Spatial preprocessing using clean-run bounds
# =========================

spatial_bounds_path = (
    CLEAN_RESULTS_DIR
    / "spatial_clip_bounds.pkl"
)

assert spatial_bounds_path.exists(), (
    f"Missing spatial clipping bounds: "
    f"{spatial_bounds_path.resolve()}"
)

spatial_clip_bounds = joblib.load(
    spatial_bounds_path
)

print("Loaded spatial clipping bounds:")
print(spatial_clip_bounds)

# Internal spatial-view order:
# 0 local_std
# 1 elev_percentile
# 2 elev_contrast
# 3 slope_variability

expected_spatial_features = [
    "local_std",
    "elev_percentile",
    "elev_contrast",
    "slope_variability",
]

assert set(expected_spatial_features).issubset(
    spatial_clip_bounds.keys()
), (
    "spatial_clip_bounds.pkl does not contain "
    "all expected spatial features."
)


def preprocess_spatial_with_clean_bounds(
    spatial_array,
    clip_bounds
):
    processed = spatial_array.astype(
        np.float32,
        copy=True
    )

    for position, feature_name in enumerate(
        expected_spatial_features
    ):
        lower = float(
            clip_bounds[feature_name]["lower"]
        )

        upper = float(
            clip_bounds[feature_name]["upper"]
        )

        processed[:, position] = np.clip(
            processed[:, position],
            lower,
            upper
        )

    if not np.isfinite(processed).all():
        raise ValueError(
            "Spatial preprocessing produced "
            "NaN or infinite values."
        )

    return processed


X_spatial_proc = preprocess_spatial_with_clean_bounds(
    X_spatial,
    spatial_clip_bounds
)

X_spatial_injected_proc = (
    preprocess_spatial_with_clean_bounds(
        X_spatial_injected,
        spatial_clip_bounds
    )
)

assert X_spatial_proc.shape == (
    n_samples,
    4
)

assert X_spatial_injected_proc.shape == (
    n_samples,
    4
)

print("Spatial preprocessing completed.")
print("Clean spatial:", X_spatial_proc.shape)
print(
    "Injected spatial:",
    X_spatial_injected_proc.shape
)

In [ ]:
# =========================
# 10. Load clean-run scalers
# =========================

scaler_paths = {
    "terrain": CLEAN_RESULTS_DIR / "terrain_scaler.pkl",
    "hydro": CLEAN_RESULTS_DIR / "hydro_scaler.pkl",
    "spatial": CLEAN_RESULTS_DIR / "spatial_scaler.pkl",
    "surface": CLEAN_RESULTS_DIR / "surface_scaler.pkl",
    "infra": CLEAN_RESULTS_DIR / "infra_scaler.pkl",
}

for scaler_name, scaler_path in scaler_paths.items():
    assert scaler_path.exists(), (
        f"Missing {scaler_name} scaler: "
        f"{scaler_path.resolve()}"
    )

terrain_scaler = joblib.load(
    scaler_paths["terrain"]
)

hydro_scaler = joblib.load(
    scaler_paths["hydro"]
)

spatial_scaler = joblib.load(
    scaler_paths["spatial"]
)

surface_scaler = joblib.load(
    scaler_paths["surface"]
)

infra_scaler = joblib.load(
    scaler_paths["infra"]
)

print("All clean-run scalers loaded.")

In [ ]:
# =========================
# 11. Transform clean and injected views
# =========================

# -------------------------
# Clean views
# -------------------------

X_terrain_scaled = terrain_scaler.transform(
    X_terrain_proc
).astype(
    np.float32,
    copy=False
)

X_hydro_scaled = hydro_scaler.transform(
    X_hydro
).astype(
    np.float32,
    copy=False
)

X_spatial_scaled = spatial_scaler.transform(
    X_spatial_proc
).astype(
    np.float32,
    copy=False
)

X_surface_scaled = surface_scaler.transform(
    X_surface
).astype(
    np.float32,
    copy=False
)

X_infra_scaled = infra_scaler.transform(
    X_infra
).astype(
    np.float32,
    copy=False
)

# -------------------------
# Injected views
# -------------------------

X_terrain_injected_scaled = terrain_scaler.transform(
    X_terrain_injected_proc
).astype(
    np.float32,
    copy=False
)

X_hydro_injected_scaled = hydro_scaler.transform(
    X_hydro_injected
).astype(
    np.float32,
    copy=False
)

X_spatial_injected_scaled = spatial_scaler.transform(
    X_spatial_injected_proc
).astype(
    np.float32,
    copy=False
)

X_surface_injected_scaled = surface_scaler.transform(
    X_surface_injected
).astype(
    np.float32,
    copy=False
)

X_infra_injected_scaled = infra_scaler.transform(
    X_infra_injected
).astype(
    np.float32,
    copy=False
)

# Land-use remains unchanged.
X_landuse_scaled = X_landuse.astype(
    np.float32,
    copy=False
)

X_landuse_injected_scaled = X_landuse_injected.astype(
    np.float32,
    copy=False
)

print("All views transformed using clean-run scalers.")

In [ ]:
scaled_clean_views = {
    "terrain": X_terrain_scaled,
    "hydro": X_hydro_scaled,
    "spatial": X_spatial_scaled,
    "surface": X_surface_scaled,
    "infra": X_infra_scaled,
    "landuse": X_landuse_scaled,
}

scaled_injected_views = {
    "terrain": X_terrain_injected_scaled,
    "hydro": X_hydro_injected_scaled,
    "spatial": X_spatial_injected_scaled,
    "surface": X_surface_injected_scaled,
    "infra": X_infra_injected_scaled,
    "landuse": X_landuse_injected_scaled,
}

for view_name in scaled_clean_views:
    clean_view = scaled_clean_views[view_name]
    injected_view = scaled_injected_views[view_name]

    assert clean_view.shape == injected_view.shape
    assert np.isfinite(clean_view).all()
    assert np.isfinite(injected_view).all()

    print(
        f"{view_name:10s} "
        f"shape={clean_view.shape}, "
        f"clean range=({clean_view.min():.4f}, "
        f"{clean_view.max():.4f}), "
        f"injected range=({injected_view.min():.4f}, "
        f"{injected_view.max():.4f})"
    )

In [ ]:
# =========================
# Verify that injection actually changed the data
# =========================

# Compare only injected rows.
row_diff = np.abs(
    X_injected[inject_idx] - X[inject_idx]
)

changed_elements = row_diff > 1e-8
changed_rows = np.any(changed_elements, axis=1)

print("Injected rows requested:", len(inject_idx))
print("Injected rows actually changed:", int(changed_rows.sum()))
print(
    "Percentage of injected rows changed:",
    changed_rows.mean() * 100
)

print(
    "Total changed feature values:",
    int(changed_elements.sum())
)

print(
    "Maximum absolute change:",
    float(row_diff.max())
)

print(
    "Mean absolute change among changed values:",
    float(row_diff[changed_elements].mean())
)

In [ ]:
# =========================
# Changed values per raw feature
# =========================

for feature_position, feature_name in enumerate(feature_names):
    feature_changed = (
        np.abs(
            X_injected[inject_idx, feature_position]
            - X[inject_idx, feature_position]
        )
        > 1e-8
    )

    count = int(feature_changed.sum())

    if count > 0:
        mean_change = float(
            np.mean(
                np.abs(
                    X_injected[
                        inject_idx[feature_changed],
                        feature_position
                    ]
                    - X[
                        inject_idx[feature_changed],
                        feature_position
                    ]
                )
            )
        )
    else:
        mean_change = 0.0

    print(
        f"{feature_name:20s} "
        f"changed rows={count:5d}, "
        f"mean abs change={mean_change:.6f}"
    )

In [ ]:
# =========================
# Verify changes after preprocessing and scaling
# =========================

scaled_view_pairs = {
    "terrain": (
        X_terrain_scaled,
        X_terrain_injected_scaled
    ),
    "hydro": (
        X_hydro_scaled,
        X_hydro_injected_scaled
    ),
    "spatial": (
        X_spatial_scaled,
        X_spatial_injected_scaled
    ),
    "surface": (
        X_surface_scaled,
        X_surface_injected_scaled
    ),
    "infra": (
        X_infra_scaled,
        X_infra_injected_scaled
    ),
    "landuse": (
        X_landuse_scaled,
        X_landuse_injected_scaled
    ),
}

for view_name, (
    clean_view,
    injected_view
) in scaled_view_pairs.items():

    difference = np.abs(
        injected_view[inject_idx]
        - clean_view[inject_idx]
    )

    changed_rows_view = np.any(
        difference > 1e-8,
        axis=1
    )

    print(
        f"{view_name:10s} "
        f"changed rows={int(changed_rows_view.sum()):5d}, "
        f"max change={float(difference.max()):.6f}, "
        f"mean change={float(difference.mean()):.6f}"
    )

In [ ]:
# =========================
# 12. PyTorch imports and device
# =========================

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

if torch.cuda.is_available():
    device = torch.device("cuda")
elif (
    hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

if device.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

In [ ]:
# =========================
# 13. Enforce float32
# =========================

X_terrain_scaled = X_terrain_scaled.astype(
    np.float32,
    copy=False
)

X_hydro_scaled = X_hydro_scaled.astype(
    np.float32,
    copy=False
)

X_spatial_scaled = X_spatial_scaled.astype(
    np.float32,
    copy=False
)

X_surface_scaled = X_surface_scaled.astype(
    np.float32,
    copy=False
)

X_infra_scaled = X_infra_scaled.astype(
    np.float32,
    copy=False
)

X_landuse_scaled = X_landuse_scaled.astype(
    np.float32,
    copy=False
)

X_terrain_injected_scaled = (
    X_terrain_injected_scaled.astype(
        np.float32,
        copy=False
    )
)

X_hydro_injected_scaled = (
    X_hydro_injected_scaled.astype(
        np.float32,
        copy=False
    )
)

X_spatial_injected_scaled = (
    X_spatial_injected_scaled.astype(
        np.float32,
        copy=False
    )
)

X_surface_injected_scaled = (
    X_surface_injected_scaled.astype(
        np.float32,
        copy=False
    )
)

X_infra_injected_scaled = (
    X_infra_injected_scaled.astype(
        np.float32,
        copy=False
    )
)

X_landuse_injected_scaled = (
    X_landuse_injected_scaled.astype(
        np.float32,
        copy=False
    )
)

print("All scaled arrays use float32.")

In [ ]:
# =========================
# 14. Construct full 23-feature reconstruction targets
# =========================

X_full_scaled = np.concatenate(
    [
        X_terrain_scaled,   # 5
        X_hydro_scaled,     # 3
        X_spatial_scaled,   # 4
        X_surface_scaled,   # 3
        X_infra_scaled,     # 1
        X_landuse_scaled,   # 7
    ],
    axis=1
).astype(
    np.float32,
    copy=False
)

X_full_injected_scaled = np.concatenate(
    [
        X_terrain_injected_scaled,  # 5
        X_hydro_injected_scaled,    # 3
        X_spatial_injected_scaled,  # 4
        X_surface_injected_scaled,  # 3
        X_infra_injected_scaled,    # 1
        X_landuse_injected_scaled,  # 7
    ],
    axis=1
).astype(
    np.float32,
    copy=False
)

expected_full_shape = (
    n_samples,
    23
)

assert X_full_scaled.shape == expected_full_shape
assert X_full_injected_scaled.shape == expected_full_shape

assert np.isfinite(X_full_scaled).all()
assert np.isfinite(
    X_full_injected_scaled
).all()

print("Clean full input:", X_full_scaled.shape)
print(
    "Injected full input:",
    X_full_injected_scaled.shape
)

In [ ]:
# =========================
# 15. Multi-view inference dataset
# =========================

class MultiViewGeoDataset(Dataset):
    def __init__(
        self,
        X_terrain,
        X_hydro,
        X_spatial,
        X_surface,
        X_infra,
        X_landuse,
        X_full
    ):
        arrays = {
            "terrain": X_terrain,
            "hydro": X_hydro,
            "spatial": X_spatial,
            "surface": X_surface,
            "infra": X_infra,
            "landuse": X_landuse,
            "full": X_full,
        }

        row_counts = {
            array.shape[0]
            for array in arrays.values()
        }

        if len(row_counts) != 1:
            raise ValueError(
                "All views must contain the same number of rows."
            )

        expected_dims = {
            "terrain": 5,
            "hydro": 3,
            "spatial": 4,
            "surface": 3,
            "infra": 1,
            "landuse": 7,
            "full": 23,
        }

        for name, array in arrays.items():
            if array.ndim != 2:
                raise ValueError(
                    f"{name} must be a two-dimensional array."
                )

            if array.shape[1] != expected_dims[name]:
                raise ValueError(
                    f"{name} expected {expected_dims[name]} "
                    f"features, found {array.shape[1]}."
                )

            if array.dtype != np.float32:
                raise ValueError(
                    f"{name} must use float32."
                )

            if not np.isfinite(array).all():
                raise ValueError(
                    f"{name} contains NaN or infinite values."
                )

        self.X_terrain = X_terrain
        self.X_hydro = X_hydro
        self.X_spatial = X_spatial
        self.X_surface = X_surface
        self.X_infra = X_infra
        self.X_landuse = X_landuse
        self.X_full = X_full

    def __len__(self):
        return self.X_full.shape[0]

    def __getitem__(self, idx):
        return (
            torch.from_numpy(
                self.X_terrain[idx]
            ),
            torch.from_numpy(
                self.X_hydro[idx]
            ),
            torch.from_numpy(
                self.X_spatial[idx]
            ),
            torch.from_numpy(
                self.X_surface[idx]
            ),
            torch.from_numpy(
                self.X_infra[idx]
            ),
            torch.from_numpy(
                self.X_landuse[idx]
            ),
            torch.from_numpy(
                self.X_full[idx]
            ),
        )

In [ ]:
# =========================
# 16. Build injected inference dataset
# =========================

dataset_injected = MultiViewGeoDataset(
    X_terrain_injected_scaled,
    X_hydro_injected_scaled,
    X_spatial_injected_scaled,
    X_surface_injected_scaled,
    X_infra_injected_scaled,
    X_landuse_injected_scaled,
    X_full_injected_scaled
)

print("Injected dataset size:", len(dataset_injected))

In [ ]:
dataset_clean = MultiViewGeoDataset(
    X_terrain_scaled,
    X_hydro_scaled,
    X_spatial_scaled,
    X_surface_scaled,
    X_infra_scaled,
    X_landuse_scaled,
    X_full_scaled
)

print("Clean dataset size:", len(dataset_clean))

In [ ]:
# =========================
# 17. Injected inference DataLoader
# =========================

inference_batch_size = 4096

loader_injected = DataLoader(
    dataset_injected,
    batch_size=inference_batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    drop_last=False
)

print("Injected loader created.")
print("Batch size:", inference_batch_size)
print("Number of batches:", len(loader_injected))

In [ ]:
# =========================
# 18. Multi-view autoencoder
# =========================

class MultiViewAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        # -------------------------
        # View-specific encoders
        # -------------------------

        # Terrain: 5 features
        self.terrain_encoder = nn.Sequential(
            nn.Linear(5, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )

        # Hydrological: 3 features
        self.hydro_encoder = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU()
        )

        # Spatial: 4 features
        self.spatial_encoder = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU()
        )

        # Surface: 3 features
        self.surface_encoder = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU()
        )

        # Infrastructure: 1 feature
        self.infra_encoder = nn.Sequential(
            nn.Linear(1, 8),
            nn.ReLU(),
            nn.Linear(8, 4),
            nn.ReLU()
        )

        # Land-use: 7 features
        self.landuse_encoder = nn.Sequential(
            nn.Linear(7, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU()
        )

        # -------------------------
        # Shared fusion embedding
        # -------------------------

        # 16 + 8 + 8 + 8 + 4 + 8 = 52
        self.fusion = nn.Sequential(
            nn.Linear(52, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )

        # -------------------------
        # Reconstruction decoder
        # -------------------------

        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 23)
        )

    def forward(
        self,
        terrain,
        hydro,
        spatial,
        surface,
        infra,
        landuse
    ):
        e_terrain = self.terrain_encoder(
            terrain
        )

        e_hydro = self.hydro_encoder(
            hydro
        )

        e_spatial = self.spatial_encoder(
            spatial
        )

        e_surface = self.surface_encoder(
            surface
        )

        e_infra = self.infra_encoder(
            infra
        )

        e_landuse = self.landuse_encoder(
            landuse
        )

        combined = torch.cat(
            [
                e_terrain,
                e_hydro,
                e_spatial,
                e_surface,
                e_infra,
                e_landuse,
            ],
            dim=1
        )

        embedding = self.fusion(
            combined
        )

        reconstruction = self.decoder(
            embedding
        )

        return reconstruction, embedding

In [ ]:
# =========================
# 19. Load trained multi-view autoencoder
# =========================

model_path = (
    CLEAN_RESULTS_DIR
    / "multiview_autoencoder_best.pth"
)

assert model_path.exists(), (
    f"Missing trained model: "
    f"{model_path.resolve()}"
)

model = MultiViewAutoencoder().to(
    device
)

checkpoint = torch.load(
    model_path,
    map_location=device
)

model.load_state_dict(
    checkpoint
)

model.eval()

print("Multi-view autoencoder loaded.")
print("Checkpoint:", model_path.resolve())
print("Model set to evaluation mode.")

In [ ]:
# =========================
# 20. Extract injected embeddings and reconstruction errors
# =========================

model.eval()

n_injected_samples = len(
    dataset_injected
)

embedding_dim = 16

embeddings_injected = np.empty(
    (
        n_injected_samples,
        embedding_dim
    ),
    dtype=np.float32
)

reconstruction_error_injected = np.empty(
    n_injected_samples,
    dtype=np.float32
)

start_idx = 0

with torch.no_grad():
    for (
        terrain,
        hydro,
        spatial,
        surface,
        infra,
        landuse,
        full
    ) in tqdm(
        loader_injected,
        desc=f"AE inference: {SCENARIO}"
    ):

        terrain = terrain.to(
            device,
            non_blocking=True
        )

        hydro = hydro.to(
            device,
            non_blocking=True
        )

        spatial = spatial.to(
            device,
            non_blocking=True
        )

        surface = surface.to(
            device,
            non_blocking=True
        )

        infra = infra.to(
            device,
            non_blocking=True
        )

        landuse = landuse.to(
            device,
            non_blocking=True
        )

        full = full.to(
            device,
            non_blocking=True
        )

        reconstruction, embedding = model(
            terrain,
            hydro,
            spatial,
            surface,
            infra,
            landuse
        )

        reconstruction_error = torch.mean(
            (
                reconstruction - full
            ) ** 2,
            dim=1
        )

        embedding_np = embedding.cpu().numpy().astype(
            np.float32,
            copy=False
        )

        error_np = (
            reconstruction_error
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=False
            )
        )

        current_batch_size = embedding_np.shape[0]
        end_idx = start_idx + current_batch_size

        embeddings_injected[
            start_idx:end_idx
        ] = embedding_np

        reconstruction_error_injected[
            start_idx:end_idx
        ] = error_np

        start_idx = end_idx

assert start_idx == n_injected_samples

assert embeddings_injected.shape == (
    n_samples,
    16
)

assert reconstruction_error_injected.shape == (
    n_samples,
)

assert np.isfinite(
    embeddings_injected
).all()

assert np.isfinite(
    reconstruction_error_injected
).all()

print(
    "Injected embeddings shape:",
    embeddings_injected.shape
)

print(
    "Injected reconstruction-error shape:",
    reconstruction_error_injected.shape
)

In [ ]:
# =========================
# 21. Save injected AE outputs
# =========================

SCENARIO_RESULTS_DIR = (
    INJECTED_RESULTS_DIR
    / SCENARIO
)

SCENARIO_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

np.save(
    SCENARIO_RESULTS_DIR
    / "embeddings.npy",
    embeddings_injected
)

np.save(
    SCENARIO_RESULTS_DIR
    / "ae_reconstruction_error_raw.npy",
    reconstruction_error_injected
)

print("Saved:")
print(
    SCENARIO_RESULTS_DIR
    / "embeddings.npy"
)

print(
    SCENARIO_RESULTS_DIR
    / "ae_reconstruction_error_raw.npy"
)

In [ ]:
# =========================
# 22. Injected AE error summary
# =========================

print(
    f"Reconstruction-error summary: {SCENARIO}"
)

print(
    "Minimum:",
    float(
        reconstruction_error_injected.min()
    )
)

print(
    "Maximum:",
    float(
        reconstruction_error_injected.max()
    )
)

print(
    "Mean:",
    float(
        reconstruction_error_injected.mean()
    )
)

print(
    "Median:",
    float(
        np.median(
            reconstruction_error_injected
        )
    )
)

for percentile in [
    95,
    98,
    99
]:
    value = np.percentile(
        reconstruction_error_injected,
        percentile
    )

    print(
        f"{percentile}th percentile:",
        float(value)
    )

In [ ]:
# =========================
# 23. Compare injected and non-injected AE errors
# =========================

injected_mask = injection_labels.astype(
    bool
)

non_injected_mask = ~injected_mask

ae_error_injected_rows = (
    reconstruction_error_injected[
        injected_mask
    ]
)

ae_error_non_injected_rows = (
    reconstruction_error_injected[
        non_injected_mask
    ]
)

print("\nAE reconstruction-error comparison")

print(
    "Injected-row mean:",
    float(
        ae_error_injected_rows.mean()
    )
)

print(
    "Non-injected-row mean:",
    float(
        ae_error_non_injected_rows.mean()
    )
)

print(
    "Injected-row median:",
    float(
        np.median(
            ae_error_injected_rows
        )
    )
)

print(
    "Non-injected-row median:",
    float(
        np.median(
            ae_error_non_injected_rows
        )
    )
)

mean_error_ratio = (
    ae_error_injected_rows.mean()
    /
    (
        ae_error_non_injected_rows.mean()
        + 1e-12
    )
)

print(
    "Injected/non-injected mean ratio:",
    float(mean_error_ratio)
)

In [ ]:
# =========================
# 24. Plot injected AE reconstruction errors
# =========================

import matplotlib.pyplot as plt

plt.figure(
    figsize=(7, 5)
)

upper_plot_limit = float(
    np.percentile(
        reconstruction_error_injected,
        99.5
    )
)

plt.hist(
    reconstruction_error_injected,
    bins=100
)

for percentile in [
    95,
    98,
    99
]:
    percentile_value = float(
        np.percentile(
            reconstruction_error_injected,
            percentile
        )
    )

    plt.axvline(
        percentile_value,
        linestyle="--",
        label=(
            f"{percentile}th: "
            f"{percentile_value:.4f}"
        )
    )

plt.xlim(
    0,
    upper_plot_limit
)

plt.xlabel(
    "Reconstruction Error"
)

plt.ylabel(
    "Frequency"
)

plt.title(
    f"AE Reconstruction Error: {SCENARIO}"
)

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

plt.hist(
    reconstruction_error_injected[non_injected_mask],
    bins=100,
    density=True,
    alpha=0.6,
    label="Non-injected"
)

plt.hist(
    reconstruction_error_injected[injected_mask],
    bins=100,
    density=True,
    alpha=0.6,
    label="Injected"
)

plt.xlabel("Reconstruction Error")
plt.ylabel("Density")
plt.title(f"AE Reconstruction Error: {SCENARIO}")
plt.yscale("log")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Hybrid anomaly detection stage: This is where our method becomes novel.

In [ ]:
# =========================
# 25. Normalize injected AE reconstruction error
#    using clean-run bounds
# =========================

ae_norm_params_path = (
    CLEAN_RESULTS_DIR
    / "ae_normalization_params.npz"
)

assert ae_norm_params_path.exists(), (
    f"Missing AE normalization parameters: "
    f"{ae_norm_params_path.resolve()}"
)

ae_norm_params = np.load(
    ae_norm_params_path
)

ae_clean_p1 = float(
    ae_norm_params["p1"]
)

ae_clean_p99 = float(
    ae_norm_params["p99"]
)

if ae_clean_p99 <= ae_clean_p1:
    raise ValueError(
        "Invalid clean AE normalization bounds."
    )

ae_score_injected_norm = (
    np.clip(
        reconstruction_error_injected,
        ae_clean_p1,
        ae_clean_p99
    )
    - ae_clean_p1
) / (
    ae_clean_p99
    - ae_clean_p1
)

# Scores outside the clean reference range remain bounded.
ae_score_injected_norm = np.clip(
    ae_score_injected_norm,
    0.0,
    1.0
).astype(
    np.float32,
    copy=False
)

assert ae_score_injected_norm.shape == (
    n_samples,
)

assert np.isfinite(
    ae_score_injected_norm
).all()

print("Injected AE normalization completed.")
print("Clean p1:", ae_clean_p1)
print("Clean p99:", ae_clean_p99)
print(
    "Minimum:",
    float(ae_score_injected_norm.min())
)
print(
    "Maximum:",
    float(ae_score_injected_norm.max())
)
print(
    "Mean:",
    float(ae_score_injected_norm.mean())
)

np.save(
    SCENARIO_RESULTS_DIR
    / "ae_score_normalized.npy",
    ae_score_injected_norm
)

print(
    "Saved:",
    SCENARIO_RESULTS_DIR
    / "ae_score_normalized.npy"
)

In [ ]:
# =========================
# 26. Create exact top-2% AE anomaly labels
# =========================

def select_exact_top_fraction(
    scores,
    fraction=0.02
):
    scores = np.asarray(scores)

    if scores.ndim != 1:
        raise ValueError(
            "Scores must be one-dimensional."
        )

    if not np.isfinite(scores).all():
        raise ValueError(
            "Scores contain NaN or infinite values."
        )

    if not 0.0 < fraction < 1.0:
        raise ValueError(
            "fraction must lie between 0 and 1."
        )

    n_select = int(
        np.ceil(
            fraction * scores.size
        )
    )

    selected_indices = np.argpartition(
        scores,
        -n_select
    )[-n_select:]

    labels = np.zeros(
        scores.size,
        dtype=np.uint8
    )

    labels[selected_indices] = 1

    threshold = float(
        scores[selected_indices].min()
    )

    return labels, threshold, selected_indices


ae_injected_labels, ae_injected_threshold, _ = (
    select_exact_top_fraction(
        ae_score_injected_norm,
        fraction=0.02
    )
)

print("AE threshold:", ae_injected_threshold)
print(
    "Total AE anomalies:",
    int(ae_injected_labels.sum())
)
print(
    "AE anomaly percentage:",
    float(ae_injected_labels.mean() * 100)
)

np.save(
    SCENARIO_RESULTS_DIR
    / "ae_anomaly_labels.npy",
    ae_injected_labels
)

print(
    "Saved:",
    SCENARIO_RESULTS_DIR
    / "ae_anomaly_labels.npy"
)

In [ ]:
# =========================
# 27. Preliminary AE injection recall
# =========================

ae_true_positives = int(
    ae_injected_labels[inject_idx].sum()
)

ae_recall = (
    ae_true_positives
    / len(inject_idx)
)

print(
    "Injected rows detected by AE:",
    ae_true_positives,
    "/",
    len(inject_idx)
)

print(
    "AE injection recall:",
    f"{ae_recall:.6f}"
)

In [ ]:
# =========================
# 28. Validate injected full feature matrix
# =========================

X_full_injected = X_full_injected_scaled

assert X_full_injected.shape == (
    n_samples,
    23
)

assert X_full_injected.dtype == np.float32

assert np.isfinite(
    X_full_injected
).all()

print(
    "Injected processed feature matrix:",
    X_full_injected.shape
)

In [ ]:
# =========================
# 29. Load clean feature-space IF model
# =========================

if_feature_model_path = (
    CLEAN_RESULTS_DIR
    / "if_feature_model.pkl"
)

assert if_feature_model_path.exists(), (
    f"Missing feature-space IF model: "
    f"{if_feature_model_path.resolve()}"
)

if_feature_model = joblib.load(
    if_feature_model_path
)

print("Feature-space IF model loaded.")
print(
    "Model:",
    if_feature_model_path.resolve()
)

In [ ]:
# =========================
# 30. Compute injected feature-space IF scores
# =========================

def compute_if_scores(
    model,
    X,
    batch_size=100_000
):
    n_rows = X.shape[0]

    scores = np.empty(
        n_rows,
        dtype=np.float32
    )

    for start in tqdm(
        range(0, n_rows, batch_size),
        desc=f"Feature IF scoring: {SCENARIO}"
    ):
        end = min(
            start + batch_size,
            n_rows
        )

        batch = X[start:end].astype(
            np.float32,
            copy=False
        )

        # Larger returned values should indicate stronger anomalies.
        batch_scores = -model.decision_function(
            batch
        )

        scores[start:end] = batch_scores.astype(
            np.float32,
            copy=False
        )

    if not np.isfinite(scores).all():
        raise ValueError(
            "Injected IF scores contain NaN or infinite values."
        )

    return scores


IF_feat_injected_raw = compute_if_scores(
    model=if_feature_model,
    X=X_full_injected,
    batch_size=100_000
)

print(
    "Injected feature-space IF scores:",
    IF_feat_injected_raw.shape
)

print(
    "Minimum:",
    float(IF_feat_injected_raw.min())
)

print(
    "Maximum:",
    float(IF_feat_injected_raw.max())
)

print(
    "Mean:",
    float(IF_feat_injected_raw.mean())
)

In [ ]:
# =========================
# 31. Normalize injected feature-space IF scores
# =========================

if_feature_norm_path = (
    CLEAN_RESULTS_DIR
    / "if_feature_normalization_params.npz"
)

assert if_feature_norm_path.exists(), (
    f"Missing IF normalization parameters: "
    f"{if_feature_norm_path.resolve()}"
)

if_feature_norm_params = np.load(
    if_feature_norm_path
)

if_feature_clean_p1 = float(
    if_feature_norm_params["p1"]
)

if_feature_clean_p99 = float(
    if_feature_norm_params["p99"]
)

if if_feature_clean_p99 <= if_feature_clean_p1:
    raise ValueError(
        "Invalid clean IF normalization bounds."
    )

IF_feat_injected_norm = (
    np.clip(
        IF_feat_injected_raw,
        if_feature_clean_p1,
        if_feature_clean_p99
    )
    - if_feature_clean_p1
) / (
    if_feature_clean_p99
    - if_feature_clean_p1
)

IF_feat_injected_norm = np.clip(
    IF_feat_injected_norm,
    0.0,
    1.0
).astype(
    np.float32,
    copy=False
)

assert IF_feat_injected_norm.shape == (
    n_samples,
)

assert np.isfinite(
    IF_feat_injected_norm
).all()

print("Injected feature-space IF normalized statistics:")
print("Clean p1:", if_feature_clean_p1)
print("Clean p99:", if_feature_clean_p99)
print(
    "Minimum:",
    float(IF_feat_injected_norm.min())
)
print(
    "Maximum:",
    float(IF_feat_injected_norm.max())
)
print(
    "Mean:",
    float(IF_feat_injected_norm.mean())
)

np.save(
    SCENARIO_RESULTS_DIR
    / "if_feature_score_normalized.npy",
    IF_feat_injected_norm
)

print(
    "Saved:",
    SCENARIO_RESULTS_DIR
    / "if_feature_score_normalized.npy"
)

In [ ]:
# =========================
# 32. Create exact top-2% feature-space IF labels
# =========================

IF_feat_injected_labels, IF_feat_injected_threshold, _ = (
    select_exact_top_fraction(
        IF_feat_injected_norm,
        fraction=0.02
    )
)

print(
    "Feature IF threshold:",
    IF_feat_injected_threshold
)

print(
    "Feature IF anomaly count:",
    int(IF_feat_injected_labels.sum())
)

print(
    "Feature IF anomaly percentage:",
    float(
        IF_feat_injected_labels.mean()
        * 100
    )
)

np.save(
    SCENARIO_RESULTS_DIR
    / "if_feature_anomaly_labels.npy",
    IF_feat_injected_labels
)

In [ ]:
if_feature_true_positives = int(
    IF_feat_injected_labels[
        inject_idx
    ].sum()
)

if_feature_recall = (
    if_feature_true_positives
    / len(inject_idx)
)

print(
    "Injected rows detected by feature IF:",
    if_feature_true_positives,
    "/",
    len(inject_idx)
)

print(
    "Feature IF injection recall:",
    f"{if_feature_recall:.6f}"
)

In [ ]:
# =========================
# 33. Load clean feature-space LOF model
# =========================

lof_feature_model_path = (
    CLEAN_RESULTS_DIR
    / "lof_feature_model.pkl"
)

assert lof_feature_model_path.exists(), (
    f"Missing feature-space LOF model: "
    f"{lof_feature_model_path.resolve()}"
)

lof_feature_model = joblib.load(
    lof_feature_model_path
)

print("Feature-space LOF model loaded.")
print(
    "Model:",
    lof_feature_model_path.resolve()
)

if hasattr(lof_feature_model, "_fit_method"):
    print(
        "Nearest-neighbor method:",
        lof_feature_model._fit_method
    )

In [ ]:
# =========================
# 34. Compute injected feature-space LOF scores
# =========================

def compute_lof_scores(
    model,
    X,
    batch_size=100_000
):
    n_rows = X.shape[0]

    scores = np.empty(
        n_rows,
        dtype=np.float32
    )

    for start in tqdm(
        range(0, n_rows, batch_size),
        desc=f"Feature LOF scoring: {SCENARIO}"
    ):
        end = min(
            start + batch_size,
            n_rows
        )

        batch = X[start:end].astype(
            np.float32,
            copy=False
        )

        # score_samples:
        # larger values indicate more normal samples.
        # Negation makes larger values indicate stronger anomalies.
        batch_scores = -model.score_samples(
            batch
        )

        scores[start:end] = (
            batch_scores.astype(
                np.float32,
                copy=False
            )
        )

    if not np.isfinite(scores).all():
        raise ValueError(
            "Injected feature-space LOF scores "
            "contain NaN or infinite values."
        )

    return scores


LOF_feat_injected_raw = compute_lof_scores(
    model=lof_feature_model,
    X=X_full_injected,
    batch_size=100_000
)

print(
    "Injected feature-space LOF scores:",
    LOF_feat_injected_raw.shape
)

print(
    "Minimum:",
    float(LOF_feat_injected_raw.min())
)

print(
    "Maximum:",
    float(LOF_feat_injected_raw.max())
)

print(
    "Mean:",
    float(LOF_feat_injected_raw.mean())
)

In [ ]:
# =========================
# 35. Inspect injected processed feature matrix
# =========================

print("Injected processed matrix shape:", X_full_injected.shape)
print("Global minimum:", float(X_full_injected.min()))
print("Global maximum:", float(X_full_injected.max()))
print("Global mean:", float(X_full_injected.mean()))
print("Global standard deviation:", float(X_full_injected.std()))

print("\nPer-feature statistics:")

for feature_position in range(
    X_full_injected.shape[1]
):
    column = X_full_injected[
        :,
        feature_position
    ]

    print(
        f"Feature {feature_position:2d}: "
        f"min={column.min():.6f}, "
        f"max={column.max():.6f}, "
        f"mean={column.mean():.6f}, "
        f"std={column.std():.6f}"
    )

In [ ]:
# =========================
# 36. Compare clean and injected processed features
# =========================

processed_difference = np.abs(
    X_full_injected[inject_idx]
    - X_full_scaled[inject_idx]
)

changed_rows_processed = np.any(
    processed_difference > 1e-8,
    axis=1
)

print(
    "Injected rows changed after preprocessing:",
    int(changed_rows_processed.sum()),
    "/",
    len(inject_idx)
)

print(
    "Maximum processed absolute change:",
    float(processed_difference.max())
)

print(
    "Mean processed absolute change:",
    float(processed_difference.mean())
)

In [ ]:
# =========================
# 37. Normalize injected feature-space LOF scores
# =========================

lof_feature_norm_path = (
    CLEAN_RESULTS_DIR
    / "lof_feature_normalization_params.npz"
)

assert lof_feature_norm_path.exists(), (
    f"Missing LOF normalization parameters: "
    f"{lof_feature_norm_path.resolve()}"
)

lof_feature_norm_params = np.load(
    lof_feature_norm_path
)

lof_feature_clean_p1 = float(
    lof_feature_norm_params["p1"]
)

lof_feature_clean_p99 = float(
    lof_feature_norm_params["p99"]
)

if lof_feature_clean_p99 <= lof_feature_clean_p1:
    raise ValueError(
        "Invalid clean LOF normalization bounds."
    )

LOF_feat_injected_norm = (
    np.clip(
        LOF_feat_injected_raw,
        lof_feature_clean_p1,
        lof_feature_clean_p99
    )
    - lof_feature_clean_p1
) / (
    lof_feature_clean_p99
    - lof_feature_clean_p1
)

LOF_feat_injected_norm = np.clip(
    LOF_feat_injected_norm,
    0.0,
    1.0
).astype(
    np.float32,
    copy=False
)

assert LOF_feat_injected_norm.shape == (
    n_samples,
)

assert np.isfinite(
    LOF_feat_injected_norm
).all()

print("Injected feature-space LOF normalized statistics:")
print("Clean p1:", lof_feature_clean_p1)
print("Clean p99:", lof_feature_clean_p99)
print(
    "Minimum:",
    float(LOF_feat_injected_norm.min())
)
print(
    "Maximum:",
    float(LOF_feat_injected_norm.max())
)
print(
    "Mean:",
    float(LOF_feat_injected_norm.mean())
)

np.save(
    SCENARIO_RESULTS_DIR
    / "lof_feature_score_normalized.npy",
    LOF_feat_injected_norm
)

print(
    "Saved:",
    SCENARIO_RESULTS_DIR
    / "lof_feature_score_normalized.npy"
)

In [ ]:
# =========================
# 38. Create exact top-2% feature-space LOF labels
# =========================

LOF_feat_injected_labels, LOF_feat_injected_threshold, _ = (
    select_exact_top_fraction(
        LOF_feat_injected_norm,
        fraction=0.02
    )
)

print(
    "Feature LOF threshold:",
    LOF_feat_injected_threshold
)

print(
    "Feature LOF anomaly count:",
    int(LOF_feat_injected_labels.sum())
)

print(
    "Feature LOF anomaly percentage:",
    float(
        LOF_feat_injected_labels.mean()
        * 100
    )
)

np.save(
    SCENARIO_RESULTS_DIR
    / "lof_feature_anomaly_labels.npy",
    LOF_feat_injected_labels
)

In [ ]:
# =========================
# 39. Preliminary feature-space LOF recall
# =========================

lof_feature_true_positives = int(
    LOF_feat_injected_labels[
        inject_idx
    ].sum()
)

lof_feature_recall = (
    lof_feature_true_positives
    / len(inject_idx)
)

print(
    "Injected rows detected by feature LOF:",
    lof_feature_true_positives,
    "/",
    len(inject_idx)
)

print(
    "Feature LOF injection recall:",
    f"{lof_feature_recall:.6f}"
)

In [ ]:
# =========================
# 40. Load clean feature-space Mahalanobis parameters
# =========================

md_feature_model_path = (
    CLEAN_RESULTS_DIR
    / "md_feature_model_params.npz"
)

assert md_feature_model_path.exists(), (
    f"Missing feature-space Mahalanobis parameters: "
    f"{md_feature_model_path.resolve()}"
)

md_feature_params = np.load(
    md_feature_model_path
)

mu_feature = md_feature_params[
    "mean"
].astype(
    np.float64,
    copy=False
)

inv_cov_feature = md_feature_params[
    "inverse_covariance"
].astype(
    np.float64,
    copy=False
)

md_feature_regularization = float(
    md_feature_params["regularization"]
)

assert mu_feature.shape == (23,)
assert inv_cov_feature.shape == (23, 23)

assert np.isfinite(mu_feature).all()
assert np.isfinite(inv_cov_feature).all()

print("Feature-space Mahalanobis parameters loaded.")
print("Mean shape:", mu_feature.shape)
print(
    "Inverse covariance shape:",
    inv_cov_feature.shape
)
print(
    "Regularization:",
    md_feature_regularization
)

In [ ]:
# =========================
# 41. Compute injected feature-space Mahalanobis scores
# =========================

def compute_mahalanobis_scores(
    X,
    mean_vector,
    inverse_covariance,
    batch_size=100_000
):
    n_rows = X.shape[0]

    scores = np.empty(
        n_rows,
        dtype=np.float32
    )

    for start in tqdm(
        range(0, n_rows, batch_size),
        desc=f"Feature MD scoring: {SCENARIO}"
    ):
        end = min(
            start + batch_size,
            n_rows
        )

        # Use float64 during the quadratic-form calculation.
        batch = X[start:end].astype(
            np.float64,
            copy=False
        )

        difference = (
            batch - mean_vector
        )

        squared_distance = np.einsum(
            "ij,jk,ik->i",
            difference,
            inverse_covariance,
            difference,
            optimize=True
        )

        # Numerical round-off may produce tiny negative values.
        squared_distance = np.maximum(
            squared_distance,
            0.0
        )

        scores[start:end] = np.sqrt(
            squared_distance
        ).astype(
            np.float32,
            copy=False
        )

    if not np.isfinite(scores).all():
        raise ValueError(
            "Injected feature-space Mahalanobis scores "
            "contain NaN or infinite values."
        )

    return scores


MD_feat_injected_raw = compute_mahalanobis_scores(
    X=X_full_injected,
    mean_vector=mu_feature,
    inverse_covariance=inv_cov_feature,
    batch_size=100_000
)

print(
    "Injected feature-space MD scores:",
    MD_feat_injected_raw.shape
)

print(
    "Minimum:",
    float(MD_feat_injected_raw.min())
)

print(
    "Maximum:",
    float(MD_feat_injected_raw.max())
)

print(
    "Mean:",
    float(MD_feat_injected_raw.mean())
)

In [ ]:
# =========================
# 42. Normalize injected feature-space MD scores
# =========================

md_feature_norm_path = (
    CLEAN_RESULTS_DIR
    / "md_feature_normalization_params.npz"
)

assert md_feature_norm_path.exists(), (
    f"Missing feature-space MD normalization parameters: "
    f"{md_feature_norm_path.resolve()}"
)

md_feature_norm_params = np.load(
    md_feature_norm_path
)

md_feature_clean_p1 = float(
    md_feature_norm_params["p1"]
)

md_feature_clean_p99 = float(
    md_feature_norm_params["p99"]
)

if md_feature_clean_p99 <= md_feature_clean_p1:
    raise ValueError(
        "Invalid clean feature-space MD normalization bounds."
    )

MD_feat_injected_norm = (
    np.clip(
        MD_feat_injected_raw,
        md_feature_clean_p1,
        md_feature_clean_p99
    )
    - md_feature_clean_p1
) / (
    md_feature_clean_p99
    - md_feature_clean_p1
)

MD_feat_injected_norm = np.clip(
    MD_feat_injected_norm,
    0.0,
    1.0
).astype(
    np.float32,
    copy=False
)

assert MD_feat_injected_norm.shape == (
    n_samples,
)

assert np.isfinite(
    MD_feat_injected_norm
).all()

print("Injected feature-space MD normalized statistics:")
print("Clean p1:", md_feature_clean_p1)
print("Clean p99:", md_feature_clean_p99)
print(
    "Minimum:",
    float(MD_feat_injected_norm.min())
)
print(
    "Maximum:",
    float(MD_feat_injected_norm.max())
)
print(
    "Mean:",
    float(MD_feat_injected_norm.mean())
)

np.save(
    SCENARIO_RESULTS_DIR
    / "md_feature_score_normalized.npy",
    MD_feat_injected_norm
)

print(
    "Saved:",
    SCENARIO_RESULTS_DIR
    / "md_feature_score_normalized.npy"
)

In [ ]:
# =========================
# 43. Create exact top-2% feature-space MD labels
# =========================

MD_feat_injected_labels, MD_feat_injected_threshold, _ = (
    select_exact_top_fraction(
        MD_feat_injected_norm,
        fraction=0.02
    )
)

print(
    "Feature MD threshold:",
    MD_feat_injected_threshold
)

print(
    "Feature MD anomaly count:",
    int(MD_feat_injected_labels.sum())
)

print(
    "Feature MD anomaly percentage:",
    float(
        MD_feat_injected_labels.mean()
        * 100
    )
)

np.save(
    SCENARIO_RESULTS_DIR
    / "md_feature_anomaly_labels.npy",
    MD_feat_injected_labels
)

In [ ]:
# =========================
# 44. Preliminary feature-space MD recall
# =========================

md_feature_true_positives = int(
    MD_feat_injected_labels[
        inject_idx
    ].sum()
)

md_feature_recall = (
    md_feature_true_positives
    / len(inject_idx)
)

print(
    "Injected rows detected by feature MD:",
    md_feature_true_positives,
    "/",
    len(inject_idx)
)

print(
    "Feature MD injection recall:",
    f"{md_feature_recall:.6f}"
)

In [ ]:
# =========================
# 45. Load clean embedding-space IF model
# =========================

if_embedding_model_path = (
    CLEAN_RESULTS_DIR
    / "if_embedding_model.pkl"
)

assert if_embedding_model_path.exists(), (
    f"Missing embedding-space IF model: "
    f"{if_embedding_model_path.resolve()}"
)

if_embedding_model = joblib.load(
    if_embedding_model_path
)

print("Embedding-space IF model loaded.")
print(
    "Model:",
    if_embedding_model_path.resolve()
)

In [ ]:
# =========================
# 46. Compute injected embedding-space IF scores
# =========================

IF_emb_injected_raw = compute_if_scores(
    model=if_embedding_model,
    X=embeddings_injected,
    batch_size=100_000
)

assert IF_emb_injected_raw.shape == (
    n_samples,
)

assert np.isfinite(
    IF_emb_injected_raw
).all()

print(
    "Injected embedding-space IF scores:",
    IF_emb_injected_raw.shape
)

print(
    "Minimum:",
    float(IF_emb_injected_raw.min())
)

print(
    "Maximum:",
    float(IF_emb_injected_raw.max())
)

print(
    "Mean:",
    float(IF_emb_injected_raw.mean())
)

In [ ]:
# =========================
# 47. Normalize injected embedding-space IF scores
# =========================

if_embedding_norm_path = (
    CLEAN_RESULTS_DIR
    / "if_embedding_normalization_params.npz"
)

assert if_embedding_norm_path.exists(), (
    f"Missing embedding-space IF normalization parameters: "
    f"{if_embedding_norm_path.resolve()}"
)

if_embedding_norm_params = np.load(
    if_embedding_norm_path
)

if_embedding_clean_p1 = float(
    if_embedding_norm_params["p1"]
)

if_embedding_clean_p99 = float(
    if_embedding_norm_params["p99"]
)

if if_embedding_clean_p99 <= if_embedding_clean_p1:
    raise ValueError(
        "Invalid clean embedding-space IF normalization bounds."
    )

IF_emb_injected_norm = (
    np.clip(
        IF_emb_injected_raw,
        if_embedding_clean_p1,
        if_embedding_clean_p99
    )
    - if_embedding_clean_p1
) / (
    if_embedding_clean_p99
    - if_embedding_clean_p1
)

IF_emb_injected_norm = np.clip(
    IF_emb_injected_norm,
    0.0,
    1.0
).astype(
    np.float32,
    copy=False
)

assert IF_emb_injected_norm.shape == (
    n_samples,
)

assert np.isfinite(
    IF_emb_injected_norm
).all()

print("Injected embedding-space IF normalized statistics:")
print("Clean p1:", if_embedding_clean_p1)
print("Clean p99:", if_embedding_clean_p99)
print(
    "Minimum:",
    float(IF_emb_injected_norm.min())
)
print(
    "Maximum:",
    float(IF_emb_injected_norm.max())
)
print(
    "Mean:",
    float(IF_emb_injected_norm.mean())
)

np.save(
    SCENARIO_RESULTS_DIR
    / "if_embedding_score_normalized.npy",
    IF_emb_injected_norm
)

print(
    "Saved:",
    SCENARIO_RESULTS_DIR
    / "if_embedding_score_normalized.npy"
)

In [ ]:
# =========================
# 48. Create exact top-2% embedding-space IF labels
# =========================

IF_emb_injected_labels, IF_emb_injected_threshold, _ = (
    select_exact_top_fraction(
        IF_emb_injected_norm,
        fraction=0.02
    )
)

print(
    "Embedding IF threshold:",
    IF_emb_injected_threshold
)

print(
    "Embedding IF anomaly count:",
    int(IF_emb_injected_labels.sum())
)

print(
    "Embedding IF anomaly percentage:",
    float(
        IF_emb_injected_labels.mean()
        * 100
    )
)

np.save(
    SCENARIO_RESULTS_DIR
    / "if_embedding_anomaly_labels.npy",
    IF_emb_injected_labels
)

In [ ]:
# =========================
# 49. Preliminary embedding-space IF recall
# =========================

if_embedding_true_positives = int(
    IF_emb_injected_labels[
        inject_idx
    ].sum()
)

if_embedding_recall = (
    if_embedding_true_positives
    / len(inject_idx)
)

print(
    "Injected rows detected by embedding IF:",
    if_embedding_true_positives,
    "/",
    len(inject_idx)
)

print(
    "Embedding IF injection recall:",
    f"{if_embedding_recall:.6f}"
)

In [ ]:
# =========================
# 50. Load clean embedding-space LOF model
# =========================

lof_embedding_model_path = (
    CLEAN_RESULTS_DIR
    / "lof_embedding_model.pkl"
)

assert lof_embedding_model_path.exists(), (
    f"Missing embedding-space LOF model: "
    f"{lof_embedding_model_path.resolve()}"
)

lof_embedding_model = joblib.load(
    lof_embedding_model_path
)

print("Embedding-space LOF model loaded.")
print(
    "Model:",
    lof_embedding_model_path.resolve()
)

if hasattr(lof_embedding_model, "_fit_method"):
    print(
        "Nearest-neighbor method:",
        lof_embedding_model._fit_method
    )

In [ ]:
# =========================
# 51. Compute injected embedding-space LOF scores
# =========================

LOF_emb_injected_raw = compute_lof_scores(
    model=lof_embedding_model,
    X=embeddings_injected,
    batch_size=100_000
)

assert LOF_emb_injected_raw.shape == (
    n_samples,
)

assert np.isfinite(
    LOF_emb_injected_raw
).all()

print(
    "Injected embedding-space LOF scores:",
    LOF_emb_injected_raw.shape
)

print(
    "Minimum:",
    float(LOF_emb_injected_raw.min())
)

print(
    "Maximum:",
    float(LOF_emb_injected_raw.max())
)

print(
    "Mean:",
    float(LOF_emb_injected_raw.mean())
)

In [ ]:
# =========================
# 52. Normalize injected embedding-space LOF scores
# =========================

lof_embedding_norm_path = (
    CLEAN_RESULTS_DIR
    / "lof_embedding_normalization_params.npz"
)

assert lof_embedding_norm_path.exists(), (
    f"Missing embedding-space LOF normalization parameters: "
    f"{lof_embedding_norm_path.resolve()}"
)

lof_embedding_norm_params = np.load(
    lof_embedding_norm_path
)

lof_embedding_clean_p1 = float(
    lof_embedding_norm_params["p1"]
)

lof_embedding_clean_p99 = float(
    lof_embedding_norm_params["p99"]
)

if lof_embedding_clean_p99 <= lof_embedding_clean_p1:
    raise ValueError(
        "Invalid clean embedding-space LOF normalization bounds."
    )

LOF_emb_injected_norm = (
    np.clip(
        LOF_emb_injected_raw,
        lof_embedding_clean_p1,
        lof_embedding_clean_p99
    )
    - lof_embedding_clean_p1
) / (
    lof_embedding_clean_p99
    - lof_embedding_clean_p1
)

LOF_emb_injected_norm = np.clip(
    LOF_emb_injected_norm,
    0.0,
    1.0
).astype(
    np.float32,
    copy=False
)

assert LOF_emb_injected_norm.shape == (
    n_samples,
)

assert np.isfinite(
    LOF_emb_injected_norm
).all()

print("Injected embedding-space LOF normalized statistics:")
print("Clean p1:", lof_embedding_clean_p1)
print("Clean p99:", lof_embedding_clean_p99)
print(
    "Minimum:",
    float(LOF_emb_injected_norm.min())
)
print(
    "Maximum:",
    float(LOF_emb_injected_norm.max())
)
print(
    "Mean:",
    float(LOF_emb_injected_norm.mean())
)

np.save(
    SCENARIO_RESULTS_DIR
    / "lof_embedding_score_normalized.npy",
    LOF_emb_injected_norm
)

print(
    "Saved:",
    SCENARIO_RESULTS_DIR
    / "lof_embedding_score_normalized.npy"
)

In [ ]:
# =========================
# 53. Create exact top-2% embedding-space LOF labels
# =========================

LOF_emb_injected_labels, LOF_emb_injected_threshold, _ = (
    select_exact_top_fraction(
        LOF_emb_injected_norm,
        fraction=0.02
    )
)

print(
    "Embedding LOF threshold:",
    LOF_emb_injected_threshold
)

print(
    "Embedding LOF anomaly count:",
    int(LOF_emb_injected_labels.sum())
)

print(
    "Embedding LOF anomaly percentage:",
    float(
        LOF_emb_injected_labels.mean()
        * 100
    )
)

np.save(
    SCENARIO_RESULTS_DIR
    / "lof_embedding_anomaly_labels.npy",
    LOF_emb_injected_labels
)

In [ ]:
# =========================
# 54. Preliminary embedding-space LOF recall
# =========================

lof_embedding_true_positives = int(
    LOF_emb_injected_labels[
        inject_idx
    ].sum()
)

lof_embedding_recall = (
    lof_embedding_true_positives
    / len(inject_idx)
)

print(
    "Injected rows detected by embedding LOF:",
    lof_embedding_true_positives,
    "/",
    len(inject_idx)
)

print(
    "Embedding LOF injection recall:",
    f"{lof_embedding_recall:.6f}"
)

In [ ]:
# =========================
# 55. Load clean embedding-space Mahalanobis parameters
# =========================

md_embedding_model_path = (
    CLEAN_RESULTS_DIR
    / "md_embedding_model_params.npz"
)

assert md_embedding_model_path.exists(), (
    f"Missing embedding-space Mahalanobis parameters: "
    f"{md_embedding_model_path.resolve()}"
)

md_embedding_params = np.load(
    md_embedding_model_path
)

mu_embedding = md_embedding_params[
    "mean"
].astype(
    np.float64,
    copy=False
)

inv_cov_embedding = md_embedding_params[
    "inverse_covariance"
].astype(
    np.float64,
    copy=False
)

md_embedding_regularization = float(
    md_embedding_params["regularization"]
)

assert mu_embedding.shape == (16,)
assert inv_cov_embedding.shape == (16, 16)

assert np.isfinite(mu_embedding).all()
assert np.isfinite(inv_cov_embedding).all()

print("Embedding-space Mahalanobis parameters loaded.")
print("Mean shape:", mu_embedding.shape)
print(
    "Inverse covariance shape:",
    inv_cov_embedding.shape
)
print(
    "Regularization:",
    md_embedding_regularization
)

In [ ]:
# =========================
# 56. Compute injected embedding-space MD scores
# =========================

MD_emb_injected_raw = compute_mahalanobis_scores(
    X=embeddings_injected,
    mean_vector=mu_embedding,
    inverse_covariance=inv_cov_embedding,
    batch_size=100_000
)

assert MD_emb_injected_raw.shape == (
    n_samples,
)

assert np.isfinite(
    MD_emb_injected_raw
).all()

print(
    "Injected embedding-space MD scores:",
    MD_emb_injected_raw.shape
)

print(
    "Minimum:",
    float(MD_emb_injected_raw.min())
)

print(
    "Maximum:",
    float(MD_emb_injected_raw.max())
)

print(
    "Mean:",
    float(MD_emb_injected_raw.mean())
)

In [ ]:
# =========================
# 57. Normalize injected embedding-space MD scores
# =========================

md_embedding_norm_path = (
    CLEAN_RESULTS_DIR
    / "md_embedding_normalization_params.npz"
)

assert md_embedding_norm_path.exists(), (
    f"Missing embedding-space MD normalization parameters: "
    f"{md_embedding_norm_path.resolve()}"
)

md_embedding_norm_params = np.load(
    md_embedding_norm_path
)

md_embedding_clean_p1 = float(
    md_embedding_norm_params["p1"]
)

md_embedding_clean_p99 = float(
    md_embedding_norm_params["p99"]
)

if md_embedding_clean_p99 <= md_embedding_clean_p1:
    raise ValueError(
        "Invalid clean embedding-space MD normalization bounds."
    )

MD_emb_injected_norm = (
    np.clip(
        MD_emb_injected_raw,
        md_embedding_clean_p1,
        md_embedding_clean_p99
    )
    - md_embedding_clean_p1
) / (
    md_embedding_clean_p99
    - md_embedding_clean_p1
)

MD_emb_injected_norm = np.clip(
    MD_emb_injected_norm,
    0.0,
    1.0
).astype(
    np.float32,
    copy=False
)

assert MD_emb_injected_norm.shape == (
    n_samples,
)

assert np.isfinite(
    MD_emb_injected_norm
).all()

print("Injected embedding-space MD normalized statistics:")
print("Clean p1:", md_embedding_clean_p1)
print("Clean p99:", md_embedding_clean_p99)
print(
    "Minimum:",
    float(MD_emb_injected_norm.min())
)
print(
    "Maximum:",
    float(MD_emb_injected_norm.max())
)
print(
    "Mean:",
    float(MD_emb_injected_norm.mean())
)

np.save(
    SCENARIO_RESULTS_DIR
    / "md_embedding_score_normalized.npy",
    MD_emb_injected_norm
)

print(
    "Saved:",
    SCENARIO_RESULTS_DIR
    / "md_embedding_score_normalized.npy"
)

In [ ]:
# =========================
# 58. Create exact top-2% embedding-space MD labels
# =========================

MD_emb_injected_labels, MD_emb_injected_threshold, _ = (
    select_exact_top_fraction(
        MD_emb_injected_norm,
        fraction=0.02
    )
)

print(
    "Embedding MD threshold:",
    MD_emb_injected_threshold
)

print(
    "Embedding MD anomaly count:",
    int(MD_emb_injected_labels.sum())
)

print(
    "Embedding MD anomaly percentage:",
    float(
        MD_emb_injected_labels.mean()
        * 100
    )
)

np.save(
    SCENARIO_RESULTS_DIR
    / "md_embedding_anomaly_labels.npy",
    MD_emb_injected_labels
)

In [ ]:
# =========================
# 59. Preliminary embedding-space MD recall
# =========================

md_embedding_true_positives = int(
    MD_emb_injected_labels[
        inject_idx
    ].sum()
)

md_embedding_recall = (
    md_embedding_true_positives
    / len(inject_idx)
)

print(
    "Injected rows detected by embedding MD:",
    md_embedding_true_positives,
    "/",
    len(inject_idx)
)

print(
    "Embedding MD injection recall:",
    f"{md_embedding_recall:.6f}"
)

In [ ]:
# =========================
# 60. Validate and summarize detector labels
# =========================

injected_score_sets = {
    "AE": ae_score_injected_norm,
    "IF feature": IF_feat_injected_norm,
    "IF embedding": IF_emb_injected_norm,
    "LOF feature": LOF_feat_injected_norm,
    "LOF embedding": LOF_emb_injected_norm,
    "MD feature": MD_feat_injected_norm,
    "MD embedding": MD_emb_injected_norm,
}

injected_label_sets = {
    "AE": ae_injected_labels,
    "IF feature": IF_feat_injected_labels,
    "IF embedding": IF_emb_injected_labels,
    "LOF feature": LOF_feat_injected_labels,
    "LOF embedding": LOF_emb_injected_labels,
    "MD feature": MD_feat_injected_labels,
    "MD embedding": MD_emb_injected_labels,
}

injected_thresholds = {
    "AE": ae_injected_threshold,
    "IF feature": IF_feat_injected_threshold,
    "IF embedding": IF_emb_injected_threshold,
    "LOF feature": LOF_feat_injected_threshold,
    "LOF embedding": LOF_emb_injected_threshold,
    "MD feature": MD_feat_injected_threshold,
    "MD embedding": MD_emb_injected_threshold,
}

expected_anomaly_count = int(
    np.ceil(
        0.02 * n_samples
    )
)

print("Detector thresholds:")

for name, threshold_value in injected_thresholds.items():
    print(
        f"{name:15s}: "
        f"{float(threshold_value):.6f}"
    )

print("\nDetector anomaly counts:")

for name, labels in injected_label_sets.items():
    assert labels.shape == (n_samples,)
    assert labels.dtype == np.uint8
    assert int(labels.sum()) == expected_anomaly_count

    print(
        f"{name:15s}: "
        f"{int(labels.sum())} "
        f"({labels.mean() * 100:.6f}%)"
    )

print("\nInjected-row recall:")

for name, labels in injected_label_sets.items():
    true_positives = int(
        labels[inject_idx].sum()
    )

    recall = (
        true_positives
        / len(inject_idx)
    )

    print(
        f"{name:15s}: "
        f"{true_positives}/{len(inject_idx)} "
        f"= {recall:.6f}"
    )

In [ ]:
# =========================
# 61. Save all injected detector labels
# =========================

np.savez_compressed(
    SCENARIO_RESULTS_DIR
    / "detector_anomaly_labels.npz",

    ae=ae_injected_labels,

    if_feature=IF_feat_injected_labels,
    if_embedding=IF_emb_injected_labels,

    lof_feature=LOF_feat_injected_labels,
    lof_embedding=LOF_emb_injected_labels,

    md_feature=MD_feat_injected_labels,
    md_embedding=MD_emb_injected_labels,
)

print(
    "Saved:",
    SCENARIO_RESULTS_DIR
    / "detector_anomaly_labels.npz"
)

In [ ]:
# =========================
# 62. Feature/embedding overlap statistics
# =========================

def compute_overlap_statistics(
    feature_labels,
    embedding_labels
):
    feature_mask = feature_labels.astype(
        bool
    )

    embedding_mask = embedding_labels.astype(
        bool
    )

    intersection = int(
        np.logical_and(
            feature_mask,
            embedding_mask
        ).sum()
    )

    union = int(
        np.logical_or(
            feature_mask,
            embedding_mask
        ).sum()
    )

    jaccard = (
        intersection / union
        if union > 0
        else 0.0
    )

    shared_fraction = (
        intersection / feature_mask.sum()
        if feature_mask.sum() > 0
        else 0.0
    )

    return (
        intersection,
        union,
        jaccard,
        shared_fraction
    )


detector_pairs = {
    "IF": (
        IF_feat_injected_labels,
        IF_emb_injected_labels
    ),
    "LOF": (
        LOF_feat_injected_labels,
        LOF_emb_injected_labels
    ),
    "MD": (
        MD_feat_injected_labels,
        MD_emb_injected_labels
    ),
}

for detector_name, (
    feature_labels,
    embedding_labels
) in detector_pairs.items():

    (
        intersection,
        union,
        jaccard,
        shared_fraction
    ) = compute_overlap_statistics(
        feature_labels,
        embedding_labels
    )

    print(f"\n{detector_name}")
    print("Intersection:", intersection)
    print("Union:", union)
    print(
        "Jaccard similarity:",
        f"{jaccard:.6f}"
    )
    print(
        "Shared fraction:",
        f"{shared_fraction * 100:.2f}%"
    )

In [ ]:
# =========================
# 63. Create injected base anomaly score
# =========================

injected_base_components = np.stack(
    [
        ae_score_injected_norm,
        IF_feat_injected_norm,
        LOF_feat_injected_norm,
        MD_feat_injected_norm,
        IF_emb_injected_norm,
        LOF_emb_injected_norm,
        MD_emb_injected_norm,
    ],
    axis=0
).astype(
    np.float32,
    copy=False
)

assert injected_base_components.shape == (
    7,
    n_samples
)

assert np.isfinite(
    injected_base_components
).all()

A_base_injected = np.mean(
    injected_base_components,
    axis=0,
    dtype=np.float32
)

A_base_injected = A_base_injected.astype(
    np.float32,
    copy=False
)

assert A_base_injected.shape == (
    n_samples,
)

assert np.isfinite(
    A_base_injected
).all()

print(
    "Injected base anomaly score ready:",
    A_base_injected.shape
)

print(
    "Minimum:",
    float(A_base_injected.min())
)

print(
    "Maximum:",
    float(A_base_injected.max())
)

print(
    "Mean:",
    float(A_base_injected.mean())
)

print(
    "Injected-row mean:",
    float(
        A_base_injected[inject_idx].mean()
    )
)

print(
    "Non-injected-row mean:",
    float(
        A_base_injected[
            injection_labels == 0
        ].mean()
    )
)

In [ ]:
# =========================
# 64. Load and validate raster mask
# =========================

mask_candidates = [
    Path("valid_mask_1.npy"),
    CLEAN_RESULTS_DIR / "valid_mask_1.npy",
]

valid_mask_path = None

for candidate in mask_candidates:
    if candidate.exists():
        valid_mask_path = candidate
        break

if valid_mask_path is None:
    raise FileNotFoundError(
        "Could not find valid_mask_1.npy either in the "
        "current directory or in "
        f"{CLEAN_RESULTS_DIR.resolve()}."
    )

valid_mask = np.load(
    valid_mask_path
).astype(
    bool,
    copy=False
)

assert valid_mask.ndim == 2

mask_pixel_count = int(
    valid_mask.sum()
)

dataset_row_count = int(
    X.shape[0]
)

print(
    "Loaded mask:",
    valid_mask_path.resolve()
)

print(
    "Valid-mask shape:",
    valid_mask.shape
)

print(
    "Mask pixels:",
    mask_pixel_count
)

print(
    "Dataset rows:",
    dataset_row_count
)

assert mask_pixel_count == dataset_row_count, (
    "The number of valid raster cells does not match "
    "the number of dataset rows."
)

print("Raster mask is consistent with the dataset.")

In [ ]:
# Stage: Spatial Consistency / Spatial Reasoning Stage
# Spatial anomaly score S(x)

In [ ]:
# =========================
# 65. Reconstruct injected base anomaly score as raster
# =========================

A_base_injected_raster = np.full(
    valid_mask.shape,
    np.nan,
    dtype=np.float32
)

A_base_injected_raster[
    valid_mask
] = A_base_injected

valid_score_count = int(
    np.isfinite(
        A_base_injected_raster
    ).sum()
)

print(
    "Injected base-score raster shape:",
    A_base_injected_raster.shape
)

print(
    "Valid anomaly-score pixels:",
    valid_score_count
)

assert valid_score_count == n_samples

assert np.isfinite(
    A_base_injected_raster[
        valid_mask
    ]
).all()

print("Injected base score reconstructed successfully.")

In [ ]:
# =========================
# 66. Compute injected spatial inconsistency
# =========================

from scipy.ndimage import uniform_filter


def nan_local_mean(
    array,
    size=5
):
    """
    Compute local mean while excluding NaN raster cells.
    """
    valid = np.isfinite(array)

    filled = np.where(
        valid,
        array,
        0.0
    ).astype(
        np.float32
    )

    kernel_area = float(
        size * size
    )

    local_sum = uniform_filter(
        filled,
        size=size,
        mode="nearest"
    ) * kernel_area

    local_count = uniform_filter(
        valid.astype(np.float32),
        size=size,
        mode="nearest"
    ) * kernel_area

    local_mean = (
        local_sum
        / np.maximum(
            local_count,
            1.0
        )
    )

    return local_mean, local_count


# Load the clean-run window size.
spatial_norm_params_path = (
    CLEAN_RESULTS_DIR
    / "spatial_score_normalization_params.npz"
)

assert spatial_norm_params_path.exists(), (
    f"Missing spatial-score normalization parameters: "
    f"{spatial_norm_params_path.resolve()}"
)

spatial_norm_params = np.load(
    spatial_norm_params_path
)

window_size = int(
    spatial_norm_params[
        "window_size"
    ]
)

print(
    "Spatial window size:",
    window_size
)

# -------------------------
# Local first moment
# -------------------------

local_mean_injected, local_count_injected = (
    nan_local_mean(
        A_base_injected_raster,
        size=window_size
    )
)

# -------------------------
# Local second moment
# -------------------------

A_base_injected_squared = np.where(
    np.isfinite(
        A_base_injected_raster
    ),
    A_base_injected_raster ** 2,
    np.nan
)

local_second_moment_injected, _ = nan_local_mean(
    A_base_injected_squared,
    size=window_size
)

# -------------------------
# Local variance and std
# -------------------------

local_variance_injected = np.maximum(
    local_second_moment_injected
    - local_mean_injected ** 2,
    0.0
)

local_std_injected = np.sqrt(
    local_variance_injected
)

# -------------------------
# Local z-score magnitude
# -------------------------

S_injected_raster = (
    np.abs(
        A_base_injected_raster
        - local_mean_injected
    )
    /
    (
        local_std_injected
        + 1e-6
    )
)

S_injected_raw = S_injected_raster[
    valid_mask
].astype(
    np.float32,
    copy=False
)

assert S_injected_raw.shape == (
    n_samples,
)

assert np.isfinite(
    S_injected_raw
).all()

print(
    "Injected spatial-score shape:",
    S_injected_raw.shape
)

print(
    "Minimum:",
    float(S_injected_raw.min())
)

print(
    "Maximum:",
    float(S_injected_raw.max())
)

print(
    "Mean:",
    float(S_injected_raw.mean())
)

In [ ]:
# =========================
# 67. Normalize injected spatial score
# =========================

spatial_clean_p1 = float(
    spatial_norm_params["p1"]
)

spatial_clean_p99 = float(
    spatial_norm_params["p99"]
)

if spatial_clean_p99 <= spatial_clean_p1:
    raise ValueError(
        "Invalid clean spatial-score normalization bounds."
    )

S_injected_norm = (
    np.clip(
        S_injected_raw,
        spatial_clean_p1,
        spatial_clean_p99
    )
    - spatial_clean_p1
) / (
    spatial_clean_p99
    - spatial_clean_p1
)

S_injected_norm = np.clip(
    S_injected_norm,
    0.0,
    1.0
).astype(
    np.float32,
    copy=False
)

assert S_injected_norm.shape == (
    n_samples,
)

assert np.isfinite(
    S_injected_norm
).all()

print("Injected normalized spatial-score statistics:")
print("Clean p1:", spatial_clean_p1)
print("Clean p99:", spatial_clean_p99)
print(
    "Minimum:",
    float(S_injected_norm.min())
)
print(
    "Maximum:",
    float(S_injected_norm.max())
)
print(
    "Mean:",
    float(S_injected_norm.mean())
)
print(
    "95th percentile:",
    float(
        np.percentile(
            S_injected_norm,
            95
        )
    )
)
print(
    "98th percentile:",
    float(
        np.percentile(
            S_injected_norm,
            98
        )
    )
)

np.save(
    SCENARIO_RESULTS_DIR
    / "spatial_score_normalized.npy",
    S_injected_norm
)

print(
    "Saved:",
    SCENARIO_RESULTS_DIR
    / "spatial_score_normalized.npy"
)

In [ ]:
# =========================
# 68. Create injected spatially refined Full score
# =========================

full_spatial_params_path = (
    CLEAN_RESULTS_DIR
    / "full_spatial_normalization_params.npz"
)

assert full_spatial_params_path.exists(), (
    f"Missing Full spatial normalization parameters: "
    f"{full_spatial_params_path.resolve()}"
)

full_spatial_params = np.load(
    full_spatial_params_path
)

full_base_weight = float(
    full_spatial_params[
        "base_weight"
    ]
)

full_spatial_weight = float(
    full_spatial_params[
        "spatial_weight"
    ]
)

A_spatial_injected = (
    full_base_weight
    * A_base_injected
    +
    full_spatial_weight
    * S_injected_norm
).astype(
    np.float32,
    copy=False
)

assert A_spatial_injected.shape == (
    n_samples,
)

assert np.isfinite(
    A_spatial_injected
).all()

print(
    "Injected spatially refined score shape:",
    A_spatial_injected.shape
)

print(
    "Base weight:",
    full_base_weight
)

print(
    "Spatial weight:",
    full_spatial_weight
)

print(
    "Minimum:",
    float(A_spatial_injected.min())
)

print(
    "Maximum:",
    float(A_spatial_injected.max())
)

print(
    "Mean:",
    float(A_spatial_injected.mean())
)

print(
    "95th percentile:",
    float(
        np.percentile(
            A_spatial_injected,
            95
        )
    )
)

print(
    "98th percentile:",
    float(
        np.percentile(
            A_spatial_injected,
            98
        )
    )
)

print(
    "99th percentile:",
    float(
        np.percentile(
            A_spatial_injected,
            99
        )
    )
)

In [ ]:
# =========================
# 69. Normalize injected spatially refined score
# =========================

a_spatial_clean_p1 = float(
    full_spatial_params["p1"]
)

a_spatial_clean_p99 = float(
    full_spatial_params["p99"]
)

if a_spatial_clean_p99 <= a_spatial_clean_p1:
    raise ValueError(
        "Invalid clean A_spatial normalization bounds."
    )

A_spatial_injected_norm = (
    np.clip(
        A_spatial_injected,
        a_spatial_clean_p1,
        a_spatial_clean_p99
    )
    - a_spatial_clean_p1
) / (
    a_spatial_clean_p99
    - a_spatial_clean_p1
)

A_spatial_injected_norm = np.clip(
    A_spatial_injected_norm,
    0.0,
    1.0
).astype(
    np.float32,
    copy=False
)

assert A_spatial_injected_norm.shape == (
    n_samples,
)

assert np.isfinite(
    A_spatial_injected_norm
).all()

print("Injected normalized spatially refined score:")
print("Clean p1:", a_spatial_clean_p1)
print("Clean p99:", a_spatial_clean_p99)
print(
    "Minimum:",
    float(A_spatial_injected_norm.min())
)
print(
    "Maximum:",
    float(A_spatial_injected_norm.max())
)
print(
    "Mean:",
    float(A_spatial_injected_norm.mean())
)

In [ ]:
np.save(
    SCENARIO_RESULTS_DIR
    / "full_spatial_score_normalized.npy",
    A_spatial_injected_norm
)

print(
    "Saved:",
    SCENARIO_RESULTS_DIR
    / "full_spatial_score_normalized.npy"
)

In [ ]:
# =========================
# 70. Create exact top-2% injected spatial labels
# =========================

S_injected_labels, S_injected_threshold, _ = (
    select_exact_top_fraction(
        S_injected_norm,
        fraction=0.02
    )
)

A_spatial_injected_labels, A_spatial_injected_threshold, _ = (
    select_exact_top_fraction(
        A_spatial_injected_norm,
        fraction=0.02
    )
)

print(
    "Spatial-signal threshold:",
    S_injected_threshold
)

print(
    "Spatial-signal anomalies:",
    int(S_injected_labels.sum())
)

print(
    "Spatially refined threshold:",
    A_spatial_injected_threshold
)

print(
    "Spatially refined anomalies:",
    int(A_spatial_injected_labels.sum())
)

In [ ]:
np.save(
    SCENARIO_RESULTS_DIR
    / "spatial_anomaly_labels.npy",
    S_injected_labels
)

In [ ]:
# =========================
# 71. Injected spatial-score percentile summary
# =========================

percentile_levels = [
    50,
    75,
    90,
    95,
    98,
    99,
]

spatial_percentiles = np.percentile(
    S_injected_norm,
    percentile_levels
)

for percentile, value in zip(
    percentile_levels,
    spatial_percentiles
):
    print(
        f"{percentile}th percentile: "
        f"{value:.4f}"
    )

In [ ]:
# Stage 6: Domain Context Score (C(x))
# So far we have: A_base → anomaly in feature space, S(x) → anomaly in spatial neighborhood. Now we add: C(x): anomaly plausibility given domain.
# We will use: Unsupervised context model (MLP Autoencoder).

In [ ]:
# =========================
# 72. Construct clean and injected context matrices
# =========================

X_context_clean = np.concatenate(
    [
        X_landuse_scaled,          # 7
        X_surface_scaled,          # 3
        X_infra_scaled,            # 1
        X_hydro_scaled,            # 3
        X_terrain_scaled[:, 0:2],  # 2: elevation and slope
    ],
    axis=1
).astype(
    np.float32,
    copy=False
)

X_context_injected = np.concatenate(
    [
        X_landuse_injected_scaled,          # 7
        X_surface_injected_scaled,          # 3
        X_infra_injected_scaled,            # 1
        X_hydro_injected_scaled,            # 3
        X_terrain_injected_scaled[:, 0:2],  # 2
    ],
    axis=1
).astype(
    np.float32,
    copy=False
)

expected_context_shape = (
    n_samples,
    16
)

assert X_context_clean.shape == (
    expected_context_shape
)

assert X_context_injected.shape == (
    expected_context_shape
)

assert np.isfinite(
    X_context_clean
).all()

assert np.isfinite(
    X_context_injected
).all()

print(
    "Clean context shape:",
    X_context_clean.shape
)

print(
    "Injected context shape:",
    X_context_injected.shape
)

In [ ]:
# =========================
# 73. Verify context changes at injected rows
# =========================

context_difference = np.abs(
    X_context_injected[inject_idx]
    - X_context_clean[inject_idx]
)

context_changed_rows = np.any(
    context_difference > 1e-8,
    axis=1
)

print(
    "Injected rows changed in context matrix:",
    int(context_changed_rows.sum()),
    "/",
    len(inject_idx)
)

print(
    "Maximum context difference:",
    float(context_difference.max())
)

print(
    "Mean context difference:",
    float(context_difference.mean())
)

In [ ]:
# =========================
# 74. Context inference dataset
# =========================

class ContextDataset(Dataset):
    def __init__(self, X):
        if X.ndim != 2:
            raise ValueError(
                "Context input must be two-dimensional."
            )

        if X.shape[1] != 16:
            raise ValueError(
                f"Expected 16 context features, "
                f"found {X.shape[1]}."
            )

        if X.dtype != np.float32:
            X = X.astype(
                np.float32,
                copy=False
            )

        if not np.isfinite(X).all():
            raise ValueError(
                "Context input contains NaN or infinite values."
            )

        self.X = X

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return torch.from_numpy(
            self.X[idx]
        )

In [ ]:
# =========================
# 75. Injected context DataLoader
# =========================

context_inference_batch_size = 4096

context_dataset_injected = ContextDataset(
    X_context_injected
)

context_loader_injected = DataLoader(
    context_dataset_injected,
    batch_size=context_inference_batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    drop_last=False
)

print(
    "Injected context dataset size:",
    len(context_dataset_injected)
)

print(
    "Context inference batches:",
    len(context_loader_injected)
)

In [ ]:
# =========================
# 76. Context autoencoder
# =========================

class ContextAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 16)
        )

    def forward(self, x):
        embedding = self.encoder(
            x
        )

        reconstruction = self.decoder(
            embedding
        )

        return reconstruction, embedding

In [ ]:
# =========================
# 77. Load clean context autoencoder
# =========================

context_model_path = (
    CLEAN_RESULTS_DIR
    / "context_autoencoder_best.pth"
)

assert context_model_path.exists(), (
    f"Missing context-autoencoder checkpoint: "
    f"{context_model_path.resolve()}"
)

model_ctx = ContextAutoencoder().to(
    device
)

context_checkpoint = torch.load(
    context_model_path,
    map_location=device
)

model_ctx.load_state_dict(
    context_checkpoint
)

model_ctx.eval()

print("Context autoencoder loaded.")
print(
    "Checkpoint:",
    context_model_path.resolve()
)
print(
    "Injected context shape:",
    X_context_injected.shape
)

In [ ]:
# =========================
# 78. Compute injected context reconstruction errors
# =========================

model_ctx.eval()

C_injected_raw = np.empty(
    n_samples,
    dtype=np.float32
)

start_idx = 0

with torch.no_grad():
    for batch in tqdm(
        context_loader_injected,
        desc=f"Context scoring: {SCENARIO}"
    ):
        batch = batch.to(
            device,
            non_blocking=True
        )

        reconstruction, _ = model_ctx(
            batch
        )

        error = torch.mean(
            (
                reconstruction - batch
            ) ** 2,
            dim=1
        )

        error_np = error.cpu().numpy().astype(
            np.float32,
            copy=False
        )

        current_batch_size = error_np.shape[0]
        end_idx = start_idx + current_batch_size

        C_injected_raw[
            start_idx:end_idx
        ] = error_np

        start_idx = end_idx

assert start_idx == n_samples

assert C_injected_raw.shape == (
    n_samples,
)

assert np.isfinite(
    C_injected_raw
).all()

print(
    "Injected context-score shape:",
    C_injected_raw.shape
)

print(
    "Minimum:",
    float(C_injected_raw.min())
)

print(
    "Maximum:",
    float(C_injected_raw.max())
)

print(
    "Mean:",
    float(C_injected_raw.mean())
)

print(
    "Median:",
    float(np.median(C_injected_raw))
)

In [ ]:
# =========================
# 79. Context-score injected/non-injected comparison
# =========================

context_injected_rows = C_injected_raw[
    inject_idx
]

context_non_injected_rows = C_injected_raw[
    injection_labels == 0
]

print("Context reconstruction-error comparison")

print(
    "Injected-row mean:",
    float(context_injected_rows.mean())
)

print(
    "Non-injected-row mean:",
    float(context_non_injected_rows.mean())
)

print(
    "Injected-row median:",
    float(np.median(context_injected_rows))
)

print(
    "Non-injected-row median:",
    float(np.median(context_non_injected_rows))
)

context_mean_ratio = (
    context_injected_rows.mean()
    /
    (
        context_non_injected_rows.mean()
        + 1e-12
    )
)

print(
    "Injected/non-injected mean ratio:",
    float(context_mean_ratio)
)

In [ ]:
# =========================
# 80. Normalize injected context score
# =========================

context_norm_params_path = (
    CLEAN_RESULTS_DIR
    / "context_score_normalization_params.npz"
)

assert context_norm_params_path.exists(), (
    f"Missing context normalization parameters: "
    f"{context_norm_params_path.resolve()}"
)

context_norm_params = np.load(
    context_norm_params_path
)

context_clean_p1 = float(
    context_norm_params["p1"]
)

context_clean_p99 = float(
    context_norm_params["p99"]
)

if context_clean_p99 <= context_clean_p1:
    raise ValueError(
        "Invalid clean context normalization bounds."
    )

C_injected_norm = (
    np.clip(
        C_injected_raw,
        context_clean_p1,
        context_clean_p99
    )
    - context_clean_p1
) / (
    context_clean_p99
    - context_clean_p1
)

C_injected_norm = np.clip(
    C_injected_norm,
    0.0,
    1.0
).astype(
    np.float32,
    copy=False
)

assert C_injected_norm.shape == (
    n_samples,
)

assert np.isfinite(
    C_injected_norm
).all()

print("Injected normalized context-score statistics:")
print("Clean p1:", context_clean_p1)
print("Clean p99:", context_clean_p99)
print(
    "Minimum:",
    float(C_injected_norm.min())
)
print(
    "Maximum:",
    float(C_injected_norm.max())
)
print(
    "Mean:",
    float(C_injected_norm.mean())
)
print(
    "95th percentile:",
    float(np.percentile(C_injected_norm, 95))
)
print(
    "98th percentile:",
    float(np.percentile(C_injected_norm, 98))
)

np.save(
    SCENARIO_RESULTS_DIR
    / "context_score_normalized.npy",
    C_injected_norm
)

print(
    "Saved:",
    SCENARIO_RESULTS_DIR
    / "context_score_normalized.npy"
)

In [ ]:
# =========================
# 81. Create exact top-2% context anomaly labels
# =========================

C_injected_labels, C_injected_threshold, _ = (
    select_exact_top_fraction(
        C_injected_norm,
        fraction=0.02
    )
)

print(
    "Context threshold:",
    C_injected_threshold
)

print(
    "Context anomaly count:",
    int(C_injected_labels.sum())
)

print(
    "Context anomaly percentage:",
    float(
        C_injected_labels.mean()
        * 100
    )
)

np.save(
    SCENARIO_RESULTS_DIR
    / "context_anomaly_labels.npy",
    C_injected_labels
)

In [ ]:
# =========================
# 82. Preliminary context-score recall
# =========================

context_true_positives = int(
    C_injected_labels[
        inject_idx
    ].sum()
)

context_recall = (
    context_true_positives
    / len(inject_idx)
)

print(
    "Injected rows detected by context score:",
    context_true_positives,
    "/",
    len(inject_idx)
)

print(
    "Context-score injection recall:",
    f"{context_recall:.6f}"
)

In [ ]:
# Stage 7: Final Fusion + Reliability Filtering
# This is our final model definition

In [ ]:
# ===============================
# 83. Full reliability signal stack
# ===============================

full_injected_score_components = {
    "IF feature": IF_feat_injected_norm,
    "LOF feature": LOF_feat_injected_norm,
    "MD feature": MD_feat_injected_norm,
    "IF embedding": IF_emb_injected_norm,
    "LOF embedding": LOF_emb_injected_norm,
    "MD embedding": MD_emb_injected_norm,
    "AE reconstruction": ae_score_injected_norm,
    "Spatial": S_injected_norm,
    "Context": C_injected_norm,
}

for signal_name, signal_scores in (
    full_injected_score_components.items()
):
    assert signal_scores.shape == (
        n_samples,
    ), (
        f"{signal_name} has shape "
        f"{signal_scores.shape}, expected {(n_samples,)}."
    )

    assert np.isfinite(
        signal_scores
    ).all(), (
        f"{signal_name} contains NaN or infinite values."
    )

score_stack_full_injected = np.stack(
    list(
        full_injected_score_components.values()
    ),
    axis=0
).astype(
    np.float32,
    copy=False
)

assert score_stack_full_injected.shape == (
    9,
    n_samples
)

print(
    "Injected Full score-stack shape:",
    score_stack_full_injected.shape
)

In [ ]:
# ===============================
# 84. Full multi-signal agreement
# ===============================

full_injected_label_components = {
    "IF feature": IF_feat_injected_labels,
    "LOF feature": LOF_feat_injected_labels,
    "MD feature": MD_feat_injected_labels,
    "IF embedding": IF_emb_injected_labels,
    "LOF embedding": LOF_emb_injected_labels,
    "MD embedding": MD_emb_injected_labels,
    "AE reconstruction": ae_injected_labels,
    "Spatial": S_injected_labels,
    "Context": C_injected_labels,
}

for signal_name, signal_labels in (
    full_injected_label_components.items()
):
    assert signal_labels.shape == (
        n_samples,
    )

    assert set(
        np.unique(signal_labels)
    ).issubset(
        {0, 1}
    ), (
        f"{signal_name} labels are not binary."
    )

multi_agreement_full_injected = np.zeros(
    n_samples,
    dtype=np.uint8
)

for signal_labels in (
    full_injected_label_components.values()
):
    multi_agreement_full_injected += (
        signal_labels.astype(
            np.uint8,
            copy=False
        )
    )

assert multi_agreement_full_injected.min() >= 0
assert multi_agreement_full_injected.max() <= 9

agreement_values, agreement_counts = np.unique(
    multi_agreement_full_injected,
    return_counts=True
)

print("Injected Full agreement distribution:")

for agreement_value, count in zip(
    agreement_values,
    agreement_counts
):
    print(
        f"Agreement {int(agreement_value)}: "
        f"{int(count)} pixels"
    )

In [ ]:
# ===============================
# 85. Compute injected Full uncertainty
# ===============================

multi_uncertainty_full_injected_raw = np.std(
    score_stack_full_injected,
    axis=0,
    dtype=np.float32
)

assert (
    multi_uncertainty_full_injected_raw.shape
    == (n_samples,)
)

assert np.isfinite(
    multi_uncertainty_full_injected_raw
).all()

print("Injected raw uncertainty statistics:")
print(
    "Minimum:",
    float(
        multi_uncertainty_full_injected_raw.min()
    )
)
print(
    "Maximum:",
    float(
        multi_uncertainty_full_injected_raw.max()
    )
)
print(
    "Mean:",
    float(
        multi_uncertainty_full_injected_raw.mean()
    )
)

In [ ]:
# ===============================
# 86. Load clean Full/Pure final parameters
# ===============================

full_pure_params_path = (
    CLEAN_RESULTS_DIR
    / "full_pure_final_normalization_params.npz"
)

assert full_pure_params_path.exists(), (
    f"Missing Full/Pure parameters: "
    f"{full_pure_params_path.resolve()}"
)

full_pure_params = np.load(
    full_pure_params_path
)

full_uncertainty_clean_min = float(
    full_pure_params[
        "full_uncertainty_min"
    ]
)

full_uncertainty_clean_max = float(
    full_pure_params[
        "full_uncertainty_max"
    ]
)

if (
    full_uncertainty_clean_max
    <= full_uncertainty_clean_min
):
    raise ValueError(
        "Invalid clean Full uncertainty bounds."
    )

print(
    "Clean Full uncertainty minimum:",
    full_uncertainty_clean_min
)

print(
    "Clean Full uncertainty maximum:",
    full_uncertainty_clean_max
)

In [ ]:
# ===============================
# 87. Normalize injected Full uncertainty
# ===============================

multi_uncertainty_full_injected_norm = (
    multi_uncertainty_full_injected_raw
    - full_uncertainty_clean_min
) / (
    full_uncertainty_clean_max
    - full_uncertainty_clean_min
)

multi_uncertainty_full_injected_norm = np.clip(
    multi_uncertainty_full_injected_norm,
    0.0,
    1.0
).astype(
    np.float32,
    copy=False
)

assert np.isfinite(
    multi_uncertainty_full_injected_norm
).all()

print("Injected normalized Full uncertainty:")
print(
    "Minimum:",
    float(
        multi_uncertainty_full_injected_norm.min()
    )
)
print(
    "Maximum:",
    float(
        multi_uncertainty_full_injected_norm.max()
    )
)
print(
    "Mean:",
    float(
        multi_uncertainty_full_injected_norm.mean()
    )
)

In [ ]:
# ===============================
# 88. Compute injected Full reliability score
# ===============================

consensus_weight = float(
    full_pure_params[
        "consensus_weight"
    ]
)

certainty_weight = float(
    full_pure_params[
        "certainty_weight"
    ]
)

full_consensus_injected = (
    multi_agreement_full_injected.astype(
        np.float32
    )
    / 9.0
)

full_certainty_injected = (
    1.0
    - multi_uncertainty_full_injected_norm
)

R_multi_full_injected = (
    consensus_weight
    * full_consensus_injected
    +
    certainty_weight
    * full_certainty_injected
).astype(
    np.float32,
    copy=False
)

assert R_multi_full_injected.shape == (
    n_samples,
)

assert np.isfinite(
    R_multi_full_injected
).all()

assert R_multi_full_injected.min() >= 0.0
assert R_multi_full_injected.max() <= 1.0

print("Injected Full reliability statistics:")
print(
    "Consensus weight:",
    consensus_weight
)
print(
    "Certainty weight:",
    certainty_weight
)
print(
    "Minimum:",
    float(R_multi_full_injected.min())
)
print(
    "Maximum:",
    float(R_multi_full_injected.max())
)
print(
    "Mean:",
    float(R_multi_full_injected.mean())
)
print(
    "95th percentile:",
    float(
        np.percentile(
            R_multi_full_injected,
            95
        )
    )
)
print(
    "99th percentile:",
    float(
        np.percentile(
            R_multi_full_injected,
            99
        )
    )
)

In [ ]:
# ===============================
# 89. Compare reliability for injected rows
# ===============================

print("Full reliability comparison:")

print(
    "Injected-row mean:",
    float(
        R_multi_full_injected[
            inject_idx
        ].mean()
    )
)

print(
    "Non-injected-row mean:",
    float(
        R_multi_full_injected[
            injection_labels == 0
        ].mean()
    )
)

print(
    "Injected-row median:",
    float(
        np.median(
            R_multi_full_injected[
                inject_idx
            ]
        )
    )
)

print(
    "Non-injected-row median:",
    float(
        np.median(
            R_multi_full_injected[
                injection_labels == 0
            ]
        )
    )
)

In [ ]:
# ===============================
# 90. Compute raw injected Full final score
# ===============================

A_final_full_injected_raw = (
    A_spatial_injected_norm
    * R_multi_full_injected
).astype(
    np.float32,
    copy=False
)

assert A_final_full_injected_raw.shape == (
    n_samples,
)

assert np.isfinite(
    A_final_full_injected_raw
).all()

print("Raw injected Full final-score statistics:")
print(
    "Minimum:",
    float(A_final_full_injected_raw.min())
)
print(
    "Maximum:",
    float(A_final_full_injected_raw.max())
)
print(
    "Mean:",
    float(A_final_full_injected_raw.mean())
)

In [ ]:
# ===============================
# 91. Normalize injected Full final score
# ===============================

full_final_clean_p1 = float(
    full_pure_params[
        "full_final_p1"
    ]
)

full_final_clean_p99 = float(
    full_pure_params[
        "full_final_p99"
    ]
)

if full_final_clean_p99 <= full_final_clean_p1:
    raise ValueError(
        "Invalid clean Full final-score normalization bounds."
    )

A_final_full_injected_norm = (
    np.clip(
        A_final_full_injected_raw,
        full_final_clean_p1,
        full_final_clean_p99
    )
    - full_final_clean_p1
) / (
    full_final_clean_p99
    - full_final_clean_p1
)

A_final_full_injected_norm = np.clip(
    A_final_full_injected_norm,
    0.0,
    1.0
).astype(
    np.float32,
    copy=False
)

assert A_final_full_injected_norm.shape == (
    n_samples,
)

assert np.isfinite(
    A_final_full_injected_norm
).all()

print("Injected Full final-score statistics:")
print("Clean p1:", full_final_clean_p1)
print("Clean p99:", full_final_clean_p99)
print(
    "Minimum:",
    float(A_final_full_injected_norm.min())
)
print(
    "Maximum:",
    float(A_final_full_injected_norm.max())
)
print(
    "Mean:",
    float(A_final_full_injected_norm.mean())
)
print(
    "95th percentile:",
    float(
        np.percentile(
            A_final_full_injected_norm,
            95
        )
    )
)
print(
    "98th percentile:",
    float(
        np.percentile(
            A_final_full_injected_norm,
            98
        )
    )
)
print(
    "99th percentile:",
    float(
        np.percentile(
            A_final_full_injected_norm,
            99
        )
    )
)

In [ ]:
# ===============================
# 92. Select exact top-2% Full anomalies
# ===============================

(
    full_injected_anomaly_labels,
    full_injected_anomaly_threshold,
    _
) = select_exact_top_fraction(
    A_final_full_injected_norm,
    fraction=0.02
)

print(
    "Injected Full threshold:",
    full_injected_anomaly_threshold
)

print(
    "Injected Full anomaly count:",
    int(
        full_injected_anomaly_labels.sum()
    )
)

print(
    "Injected Full anomaly percentage:",
    float(
        full_injected_anomaly_labels.mean()
        * 100
    )
)

In [ ]:
# ===============================
# 93. Full injected-anomaly detection results
# ===============================

full_true_positives = int(
    full_injected_anomaly_labels[
        inject_idx
    ].sum()
)

full_false_negatives = (
    len(inject_idx)
    - full_true_positives
)

full_total_predicted = int(
    full_injected_anomaly_labels.sum()
)

full_false_positives = (
    full_total_predicted
    - full_true_positives
)

full_recall = (
    full_true_positives
    / len(inject_idx)
)

full_precision = (
    full_true_positives
    / full_total_predicted
    if full_total_predicted > 0
    else 0.0
)

full_f1 = (
    2.0
    * full_precision
    * full_recall
    / (
        full_precision
        + full_recall
    )
    if (
        full_precision
        + full_recall
    ) > 0
    else 0.0
)

print("Full detection performance:")
print(
    "True positives:",
    full_true_positives
)
print(
    "False positives:",
    full_false_positives
)
print(
    "False negatives:",
    full_false_negatives
)
print(
    "Precision:",
    f"{full_precision:.6f}"
)
print(
    "Recall:",
    f"{full_recall:.6f}"
)
print(
    "F1-score:",
    f"{full_f1:.6f}"
)

In [ ]:
# ===============================
# Injection-focused evaluation
# ===============================

total_injected = len(inject_idx)

detected_injected = int(
    full_injected_anomaly_labels[inject_idx].sum()
)

missed_injected = (
    total_injected - detected_injected
)

detection_rate = (
    detected_injected / total_injected
)

miss_rate = (
    missed_injected / total_injected
)

print("Full injected-anomaly evaluation:")
print("Total injected anomalies:", total_injected)
print("Detected injected anomalies:", detected_injected)
print("Missed injected anomalies:", missed_injected)
print(
    "Injection detection rate:",
    f"{detection_rate * 100:.2f}%"
)
print(
    "Injection miss rate:",
    f"{miss_rate * 100:.2f}%"
)

In [ ]:
# ===============================
# 94. Save injected Full outputs
# ===============================

np.save(
    SCENARIO_RESULTS_DIR
    / "full_final_anomaly_score.npy",
    A_final_full_injected_norm
)

np.save(
    SCENARIO_RESULTS_DIR
    / "full_final_anomaly_labels.npy",
    full_injected_anomaly_labels.astype(
        np.uint8,
        copy=False
    )
)

np.savez_compressed(
    SCENARIO_RESULTS_DIR
    / "full_reliability_diagnostics.npz",

    agreement=multi_agreement_full_injected,

    uncertainty_raw=(
        multi_uncertainty_full_injected_raw
    ),

    uncertainty_normalized=(
        multi_uncertainty_full_injected_norm
    ),

    reliability=R_multi_full_injected,
)

print("Saved injected Full outputs:")
print(
    SCENARIO_RESULTS_DIR
    / "full_final_anomaly_score.npy"
)
print(
    SCENARIO_RESULTS_DIR
    / "full_final_anomaly_labels.npy"
)
print(
    SCENARIO_RESULTS_DIR
    / "full_reliability_diagnostics.npz"
)

In [ ]:
# ===============================
# 95. Create injected Pure spatial score
# ===============================

# Clean Pure weights
pure_base_weight = float(
    full_pure_params["pure_base_weight"]
)

pure_spatial_weight = float(
    full_pure_params["pure_spatial_weight"]
)

# Pure variant starts from the AE reconstruction signal.
A_base_pure_injected = ae_score_injected_norm

A_spatial_pure_injected = (
    pure_base_weight
    * A_base_pure_injected
    +
    pure_spatial_weight
    * S_injected_norm
).astype(
    np.float32,
    copy=False
)

assert A_spatial_pure_injected.shape == (
    n_samples,
)

assert np.isfinite(
    A_spatial_pure_injected
).all()

print("Injected Pure spatial score:")
print("AE weight:", pure_base_weight)
print("Spatial weight:", pure_spatial_weight)
print(
    "Minimum:",
    float(A_spatial_pure_injected.min())
)
print(
    "Maximum:",
    float(A_spatial_pure_injected.max())
)
print(
    "Mean:",
    float(A_spatial_pure_injected.mean())
)

In [ ]:
# ===============================
# 96. Normalize injected Pure spatial score
# ===============================

pure_spatial_clean_p1 = float(
    full_pure_params["pure_spatial_p1"]
)

pure_spatial_clean_p99 = float(
    full_pure_params["pure_spatial_p99"]
)

if pure_spatial_clean_p99 <= pure_spatial_clean_p1:
    raise ValueError(
        "Invalid clean Pure spatial normalization bounds."
    )

A_spatial_pure_injected_norm = (
    np.clip(
        A_spatial_pure_injected,
        pure_spatial_clean_p1,
        pure_spatial_clean_p99
    )
    - pure_spatial_clean_p1
) / (
    pure_spatial_clean_p99
    - pure_spatial_clean_p1
)

A_spatial_pure_injected_norm = np.clip(
    A_spatial_pure_injected_norm,
    0.0,
    1.0
).astype(
    np.float32,
    copy=False
)

assert np.isfinite(
    A_spatial_pure_injected_norm
).all()

print("Normalized injected Pure spatial score:")
print("Clean p1:", pure_spatial_clean_p1)
print("Clean p99:", pure_spatial_clean_p99)
print(
    "Minimum:",
    float(A_spatial_pure_injected_norm.min())
)
print(
    "Maximum:",
    float(A_spatial_pure_injected_norm.max())
)
print(
    "Mean:",
    float(A_spatial_pure_injected_norm.mean())
)

In [ ]:
# ===============================
# 97. Create injected Pure signal stack
# ===============================

score_stack_pure_injected = np.stack(
    [
        ae_score_injected_norm,
        S_injected_norm,
        C_injected_norm,
    ],
    axis=0
).astype(
    np.float32,
    copy=False
)

assert score_stack_pure_injected.shape == (
    3,
    n_samples
)

assert np.isfinite(
    score_stack_pure_injected
).all()

print(
    "Injected Pure score-stack shape:",
    score_stack_pure_injected.shape
)

In [ ]:
# ===============================
# 98. Compute injected Pure agreement
# ===============================

multi_agreement_pure_injected = (
    ae_injected_labels.astype(
        np.uint8,
        copy=False
    )
    +
    S_injected_labels.astype(
        np.uint8,
        copy=False
    )
    +
    C_injected_labels.astype(
        np.uint8,
        copy=False
    )
)

assert multi_agreement_pure_injected.shape == (
    n_samples,
)

assert multi_agreement_pure_injected.min() >= 0
assert multi_agreement_pure_injected.max() <= 3

pure_agreement_values, pure_agreement_counts = np.unique(
    multi_agreement_pure_injected,
    return_counts=True
)

print("Injected Pure agreement distribution:")

for agreement_value, count in zip(
    pure_agreement_values,
    pure_agreement_counts
):
    print(
        f"Agreement {int(agreement_value)}: "
        f"{int(count)} pixels"
    )

In [ ]:
# ===============================
# 99. Compute injected Pure uncertainty
# ===============================

multi_uncertainty_pure_injected_raw = np.std(
    score_stack_pure_injected,
    axis=0,
    dtype=np.float32
)

assert (
    multi_uncertainty_pure_injected_raw.shape
    == (n_samples,)
)

assert np.isfinite(
    multi_uncertainty_pure_injected_raw
).all()

print("Raw injected Pure uncertainty:")
print(
    "Minimum:",
    float(
        multi_uncertainty_pure_injected_raw.min()
    )
)
print(
    "Maximum:",
    float(
        multi_uncertainty_pure_injected_raw.max()
    )
)
print(
    "Mean:",
    float(
        multi_uncertainty_pure_injected_raw.mean()
    )
)

In [ ]:
# ===============================
# 100. Normalize injected Pure uncertainty
# ===============================

pure_uncertainty_clean_min = float(
    full_pure_params[
        "pure_uncertainty_min"
    ]
)

pure_uncertainty_clean_max = float(
    full_pure_params[
        "pure_uncertainty_max"
    ]
)

if (
    pure_uncertainty_clean_max
    <= pure_uncertainty_clean_min
):
    raise ValueError(
        "Invalid clean Pure uncertainty bounds."
    )

multi_uncertainty_pure_injected_norm = (
    multi_uncertainty_pure_injected_raw
    - pure_uncertainty_clean_min
) / (
    pure_uncertainty_clean_max
    - pure_uncertainty_clean_min
)

multi_uncertainty_pure_injected_norm = np.clip(
    multi_uncertainty_pure_injected_norm,
    0.0,
    1.0
).astype(
    np.float32,
    copy=False
)

assert np.isfinite(
    multi_uncertainty_pure_injected_norm
).all()

print("Normalized injected Pure uncertainty:")
print(
    "Minimum:",
    float(
        multi_uncertainty_pure_injected_norm.min()
    )
)
print(
    "Maximum:",
    float(
        multi_uncertainty_pure_injected_norm.max()
    )
)
print(
    "Mean:",
    float(
        multi_uncertainty_pure_injected_norm.mean()
    )
)

In [ ]:
# ===============================
# 101. Compute injected Pure reliability
# ===============================

pure_consensus_injected = (
    multi_agreement_pure_injected.astype(
        np.float32
    )
    / 3.0
)

pure_certainty_injected = (
    1.0
    - multi_uncertainty_pure_injected_norm
)

R_multi_pure_injected = (
    consensus_weight
    * pure_consensus_injected
    +
    certainty_weight
    * pure_certainty_injected
).astype(
    np.float32,
    copy=False
)

assert R_multi_pure_injected.shape == (
    n_samples,
)

assert np.isfinite(
    R_multi_pure_injected
).all()

assert R_multi_pure_injected.min() >= 0.0
assert R_multi_pure_injected.max() <= 1.0

print("Injected Pure reliability statistics:")
print(
    "Minimum:",
    float(R_multi_pure_injected.min())
)
print(
    "Maximum:",
    float(R_multi_pure_injected.max())
)
print(
    "Mean:",
    float(R_multi_pure_injected.mean())
)
print(
    "95th percentile:",
    float(
        np.percentile(
            R_multi_pure_injected,
            95
        )
    )
)
print(
    "99th percentile:",
    float(
        np.percentile(
            R_multi_pure_injected,
            99
        )
    )
)

In [ ]:
# ===============================
# 102. Compute raw injected Pure final score
# ===============================

A_final_pure_injected_raw = (
    A_spatial_pure_injected_norm
    * R_multi_pure_injected
).astype(
    np.float32,
    copy=False
)

assert A_final_pure_injected_raw.shape == (
    n_samples,
)

assert np.isfinite(
    A_final_pure_injected_raw
).all()

print("Raw injected Pure final score:")
print(
    "Minimum:",
    float(A_final_pure_injected_raw.min())
)
print(
    "Maximum:",
    float(A_final_pure_injected_raw.max())
)
print(
    "Mean:",
    float(A_final_pure_injected_raw.mean())
)

In [ ]:
# ===============================
# 103. Normalize injected Pure final score
# ===============================

pure_final_clean_p1 = float(
    full_pure_params["pure_final_p1"]
)

pure_final_clean_p99 = float(
    full_pure_params["pure_final_p99"]
)

if pure_final_clean_p99 <= pure_final_clean_p1:
    raise ValueError(
        "Invalid clean Pure final-score normalization bounds."
    )

A_final_pure_injected_norm = (
    np.clip(
        A_final_pure_injected_raw,
        pure_final_clean_p1,
        pure_final_clean_p99
    )
    - pure_final_clean_p1
) / (
    pure_final_clean_p99
    - pure_final_clean_p1
)

A_final_pure_injected_norm = np.clip(
    A_final_pure_injected_norm,
    0.0,
    1.0
).astype(
    np.float32,
    copy=False
)

assert np.isfinite(
    A_final_pure_injected_norm
).all()

print("Injected Pure final-score statistics:")
print("Clean p1:", pure_final_clean_p1)
print("Clean p99:", pure_final_clean_p99)
print(
    "Minimum:",
    float(A_final_pure_injected_norm.min())
)
print(
    "Maximum:",
    float(A_final_pure_injected_norm.max())
)
print(
    "Mean:",
    float(A_final_pure_injected_norm.mean())
)
print(
    "95th percentile:",
    float(
        np.percentile(
            A_final_pure_injected_norm,
            95
        )
    )
)
print(
    "98th percentile:",
    float(
        np.percentile(
            A_final_pure_injected_norm,
            98
        )
    )
)
print(
    "99th percentile:",
    float(
        np.percentile(
            A_final_pure_injected_norm,
            99
        )
    )
)

In [ ]:
# ===============================
# 104. Select exact top-2% Pure anomalies
# ===============================

(
    pure_injected_anomaly_labels,
    pure_injected_anomaly_threshold,
    _
) = select_exact_top_fraction(
    A_final_pure_injected_norm,
    fraction=0.02
)

print(
    "Injected Pure threshold:",
    pure_injected_anomaly_threshold
)

print(
    "Injected Pure anomaly count:",
    int(
        pure_injected_anomaly_labels.sum()
    )
)

print(
    "Injected Pure anomaly percentage:",
    float(
        pure_injected_anomaly_labels.mean()
        * 100
    )
)

In [ ]:
# ===============================
# 105. Save injected Pure outputs
# ===============================

np.save(
    SCENARIO_RESULTS_DIR
    / "pure_final_anomaly_score.npy",
    A_final_pure_injected_norm
)

np.save(
    SCENARIO_RESULTS_DIR
    / "pure_final_anomaly_labels.npy",
    pure_injected_anomaly_labels.astype(
        np.uint8,
        copy=False
    )
)

np.savez_compressed(
    SCENARIO_RESULTS_DIR
    / "pure_reliability_diagnostics.npz",

    agreement=multi_agreement_pure_injected,

    uncertainty_raw=(
        multi_uncertainty_pure_injected_raw
    ),

    uncertainty_normalized=(
        multi_uncertainty_pure_injected_norm
    ),

    reliability=R_multi_pure_injected,
)

print("Saved injected Pure outputs:")
print(
    SCENARIO_RESULTS_DIR
    / "pure_final_anomaly_score.npy"
)
print(
    SCENARIO_RESULTS_DIR
    / "pure_final_anomaly_labels.npy"
)
print(
    SCENARIO_RESULTS_DIR
    / "pure_reliability_diagnostics.npz"
)

In [ ]:
# ===============================
# 106. Evaluate known injected anomalies only
# ===============================

def evaluate_injected_detection(
    anomaly_labels,
    injected_indices
):
    total_injected = len(
        injected_indices
    )

    detected_injected = int(
        anomaly_labels[
            injected_indices
        ].sum()
    )

    missed_injected = (
        total_injected
        - detected_injected
    )

    detection_rate = (
        detected_injected
        / total_injected
    )

    miss_rate = (
        missed_injected
        / total_injected
    )

    return {
        "total_injected": total_injected,
        "detected_injected": detected_injected,
        "missed_injected": missed_injected,
        "detection_rate": detection_rate,
        "miss_rate": miss_rate,
    }


evaluation_methods = {
    "Full": full_injected_anomaly_labels,
    "Pure": pure_injected_anomaly_labels,
}

evaluation_results = {}

for method_name, anomaly_labels in (
    evaluation_methods.items()
):
    results = evaluate_injected_detection(
        anomaly_labels,
        inject_idx
    )

    evaluation_results[
        method_name
    ] = results

    print("\n==============================")
    print("Method:", method_name)
    print(
        "Total injected anomalies:",
        results["total_injected"]
    )
    print(
        "Detected injected anomalies:",
        results["detected_injected"]
    )
    print(
        "Missed injected anomalies:",
        results["missed_injected"]
    )
    print(
        "Injection detection rate:",
        f"{results['detection_rate'] * 100:.2f}%"
    )
    print(
        "Injection miss rate:",
        f"{results['miss_rate'] * 100:.2f}%"
    )

In [ ]:
# ===============================
# 107. Save injection evaluation summary
# ===============================

evaluation_summary = pd.DataFrame(
    [
        {
            "scenario": SCENARIO,
            "method": method_name,
            "total_injected": results[
                "total_injected"
            ],
            "detected_injected": results[
                "detected_injected"
            ],
            "missed_injected": results[
                "missed_injected"
            ],
            "detection_rate": results[
                "detection_rate"
            ],
            "miss_rate": results[
                "miss_rate"
            ],
        }
        for method_name, results
        in evaluation_results.items()
    ]
)

evaluation_summary.to_csv(
    SCENARIO_RESULTS_DIR
    / "injection_detection_summary.csv",
    index=False
)

print("\nEvaluation summary:")
print(evaluation_summary)

print(
    "\nSaved:",
    SCENARIO_RESULTS_DIR
    / "injection_detection_summary.csv"
)

In [ ]:
# ===============================
# Detection rate at multiple top-score budgets
# ===============================

selection_fractions = [
    0.001,   # top 0.1%
    0.0025,  # top 0.25%
    0.005,   # top 0.5%
    0.010,   # top 1%
    0.020,   # top 2%
]


def detection_at_fraction(
    scores,
    injected_indices,
    fraction
):
    labels, threshold, _ = (
        select_exact_top_fraction(
            scores,
            fraction=fraction
        )
    )

    selected_count = int(
        labels.sum()
    )

    detected_count = int(
        labels[injected_indices].sum()
    )

    total_injected = len(
        injected_indices
    )

    return {
        "fraction": float(fraction),
        "threshold": float(threshold),
        "selected_count": selected_count,
        "detected_count": detected_count,
        "missed_count": (
            total_injected - detected_count
        ),
        "detection_rate": float(
            detected_count / total_injected
        ),
    }

In [ ]:
budget_results = []

methods = {
    "Full": A_final_full_injected_norm,
    "Pure": A_final_pure_injected_norm,
}

for method_name, final_scores in methods.items():

    print("\n==============================")
    print("Method:", method_name)

    for fraction in selection_fractions:

        result = detection_at_fraction(
            scores=final_scores,
            injected_indices=inject_idx,
            fraction=fraction
        )

        result["method"] = method_name
        result["scenario"] = SCENARIO

        budget_results.append(result)

        print(
            f"Top {fraction * 100:5.2f}% | "
            f"Selected: {result['selected_count']:7d} | "
            f"Detected: {result['detected_count']:4d}/"
            f"{len(inject_idx)} | "
            f"Detection rate: "
            f"{result['detection_rate'] * 100:6.2f}%"
        )

In [ ]:
# ============================================================
# Detection rates for all internal and final methods
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Selection budgets to evaluate
# ------------------------------------------------------------

selection_fractions = [
    0.001,   # top 0.10%
    0.0025,  # top 0.25%
    0.005,   # top 0.50%
    0.010,   # top 1.00%
    0.020,   # top 2.00%
]


# ------------------------------------------------------------
# 2. All internal and final anomaly scores
# ------------------------------------------------------------

method_scores = {
    # Main autoencoder signal
    "AE": ae_score_injected_norm,

    # Feature-space detectors
    "IF_Feature": IF_feat_injected_norm,
    "LOF_Feature": LOF_feat_injected_norm,
    "MD_Feature": MD_feat_injected_norm,

    # Embedding-space detectors
    "IF_Embedding": IF_emb_injected_norm,
    "LOF_Embedding": LOF_emb_injected_norm,
    "MD_Embedding": MD_emb_injected_norm,

    # Spatial and context signals
    "Spatial": S_injected_norm,
    "Context": C_injected_norm,

    # Spatially refined intermediate scores
    "Full_Spatial": A_spatial_injected_norm,
    "Pure_Spatial": A_spatial_pure_injected_norm,

    # Final proposed variants
    "Full_Final": A_final_full_injected_norm,
    "Pure_Final": A_final_pure_injected_norm,
}


# ------------------------------------------------------------
# 3. Validate score arrays
# ------------------------------------------------------------

n_total = len(
    next(iter(method_scores.values()))
)

inject_idx_eval = np.asarray(
    inject_idx,
    dtype=np.int64
)

assert inject_idx_eval.ndim == 1
assert len(inject_idx_eval) > 0
assert inject_idx_eval.min() >= 0
assert inject_idx_eval.max() < n_total

for method_name, scores in method_scores.items():

    scores = np.asarray(scores)

    if scores.shape != (n_total,):
        raise ValueError(
            f"{method_name} has shape {scores.shape}; "
            f"expected {(n_total,)}."
        )

    if not np.isfinite(scores).all():
        raise ValueError(
            f"{method_name} contains NaN or infinite values."
        )


# ------------------------------------------------------------
# 4. Evaluate injected IDs at one selection fraction
# ------------------------------------------------------------

def evaluate_detection_at_fraction(
    scores,
    injected_indices,
    fraction
):
    """
    Select the exact highest-scoring fraction of observations and
    calculate how many known injected IDs occur in that selected set.
    """

    labels, threshold, _ = select_exact_top_fraction(
        scores,
        fraction=fraction
    )

    labels = np.asarray(
        labels,
        dtype=np.uint8
    )

    selected_count = int(
        labels.sum()
    )

    detected_injected = int(
        labels[injected_indices].sum()
    )

    total_injected = int(
        len(injected_indices)
    )

    missed_injected = (
        total_injected
        - detected_injected
    )

    detection_rate = (
        detected_injected
        / total_injected
    )

    miss_rate = (
        missed_injected
        / total_injected
    )

    return {
        "selection_fraction": float(fraction),
        "selection_percentage": float(
            fraction * 100
        ),
        "threshold": float(threshold),
        "selected_count": selected_count,
        "total_injected": total_injected,
        "detected_injected": detected_injected,
        "missed_injected": missed_injected,
        "detection_rate": float(detection_rate),
        "detection_rate_percent": float(
            detection_rate * 100
        ),
        "miss_rate": float(miss_rate),
        "miss_rate_percent": float(
            miss_rate * 100
        ),
    }


# ------------------------------------------------------------
# 5. Evaluate every method at every selection budget
# ------------------------------------------------------------

all_detection_results = []

for method_name, scores in method_scores.items():

    for fraction in selection_fractions:

        result = evaluate_detection_at_fraction(
            scores=scores,
            injected_indices=inject_idx_eval,
            fraction=fraction
        )

        result["scenario"] = SCENARIO
        result["method"] = method_name

        all_detection_results.append(
            result
        )


detection_results_df = pd.DataFrame(
    all_detection_results
)

# Arrange columns
detection_results_df = detection_results_df[
    [
        "scenario",
        "method",
        "selection_fraction",
        "selection_percentage",
        "selected_count",
        "total_injected",
        "detected_injected",
        "missed_injected",
        "detection_rate",
        "detection_rate_percent",
        "miss_rate",
        "miss_rate_percent",
        "threshold",
    ]
]


# ------------------------------------------------------------
# 6. Print detailed results
# ------------------------------------------------------------

for method_name in method_scores:

    method_results = detection_results_df[
        detection_results_df["method"]
        == method_name
    ]

    print("\n" + "=" * 76)
    print("Method:", method_name)
    print("=" * 76)

    for _, row in method_results.iterrows():

        print(
            f"Top {row['selection_percentage']:5.2f}% | "
            f"Selected: {int(row['selected_count']):7d} | "
            f"Detected: {int(row['detected_injected']):4d}/"
            f"{int(row['total_injected']):4d} | "
            f"Missed: {int(row['missed_injected']):4d} | "
            f"Detection rate: "
            f"{row['detection_rate_percent']:6.2f}%"
        )

In [ ]:
# ============================================================
# Compact detection-rate comparison table
# ============================================================

detection_rate_table = detection_results_df.pivot(
    index="method",
    columns="selection_percentage",
    values="detection_rate_percent"
)

detection_rate_table = detection_rate_table.rename(
    columns=lambda value: f"Top_{value:g}%"
)

# Desired display order
method_order = [
    "AE",
    "IF_Feature",
    "LOF_Feature",
    "MD_Feature",
    "IF_Embedding",
    "LOF_Embedding",
    "MD_Embedding",
    "Spatial",
    "Context",
    "Full_Spatial",
    "Pure_Spatial",
    "Full_Final",
    "Pure_Final",
]

detection_rate_table = detection_rate_table.reindex(
    method_order
)

print("\nDetection rate (%) by selection budget:")
print(
    detection_rate_table.round(2)
)

In [ ]:
# ============================================================
# Create top-2% ranking and save both result tables to Excel
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Validate required variables
# ------------------------------------------------------------

required_variables = [
    "detection_results_df",
    "detection_rate_table",
    "SCENARIO_RESULTS_DIR",
    "SCENARIO",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Missing required variables: "
        + ", ".join(missing_variables)
    )

# Ensure output folder exists
SCENARIO_RESULTS_DIR = Path(
    SCENARIO_RESULTS_DIR
)

SCENARIO_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Create top-2% ranking from detailed results
# ------------------------------------------------------------

top_2_results = (
    detection_results_df[
        np.isclose(
            detection_results_df[
                "selection_fraction"
            ].astype(float),
            0.02
        )
    ]
    .sort_values(
        by="detection_rate",
        ascending=False
    )
    .reset_index(drop=True)
)

if top_2_results.empty:
    raise ValueError(
        "No rows were found for selection_fraction = 0.02."
    )

top_2_results.insert(
    0,
    "rank",
    np.arange(
        1,
        len(top_2_results) + 1
    )
)

print("Top-2% ranking created:")
print(
    top_2_results[
        [
            "rank",
            "method",
            "detected_injected",
            "missed_injected",
            "detection_rate_percent",
        ]
    ]
)

# ------------------------------------------------------------
# 3. Prepare compact detection-rate table
# ------------------------------------------------------------

compact_export_df = (
    detection_rate_table
    .reset_index()
)

compact_export_df = compact_export_df.rename(
    columns={
        "method": "Method",
        "index": "Method",
    }
)

# ------------------------------------------------------------
# 4. Prepare top-2% ranking table
# ------------------------------------------------------------

top_2_export_df = top_2_results[
    [
        "rank",
        "method",
        "selected_count",
        "total_injected",
        "detected_injected",
        "missed_injected",
        "detection_rate_percent",
        "threshold",
    ]
].copy()

top_2_export_df = top_2_export_df.rename(
    columns={
        "rank": "Rank",
        "method": "Method",
        "selected_count": "Selected Pixels",
        "total_injected": "Total Injected",
        "detected_injected": "Detected Injected",
        "missed_injected": "Missed Injected",
        "detection_rate_percent": "Detection Rate (%)",
        "threshold": "Score Threshold",
    }
)

# ------------------------------------------------------------
# 5. Excel output path
# ------------------------------------------------------------

excel_output_path = (
    SCENARIO_RESULTS_DIR
    / f"{SCENARIO}_internal_detection_results.xlsx"
)

# ------------------------------------------------------------
# 6. Save both tables to Excel
# ------------------------------------------------------------

with pd.ExcelWriter(
    excel_output_path,
    engine="openpyxl"
) as writer:

    compact_export_df.to_excel(
        writer,
        sheet_name="Detection by Budget",
        index=False
    )

    top_2_export_df.to_excel(
        writer,
        sheet_name="Top 2 Percent Ranking",
        index=False
    )

    workbook = writer.book

    # --------------------------------------------------------
    # Basic formatting
    # --------------------------------------------------------

    from openpyxl.styles import (
        Font,
        PatternFill,
        Alignment,
        Border,
        Side,
    )

    header_fill = PatternFill(
        fill_type="solid",
        fgColor="D9EAF7"
    )

    thin_border = Border(
        left=Side(
            style="thin",
            color="D9D9D9"
        ),
        right=Side(
            style="thin",
            color="D9D9D9"
        ),
        top=Side(
            style="thin",
            color="D9D9D9"
        ),
        bottom=Side(
            style="thin",
            color="D9D9D9"
        ),
    )

    for sheet_name in workbook.sheetnames:

        worksheet = workbook[
            sheet_name
        ]

        # Header formatting
        for cell in worksheet[1]:
            cell.fill = header_fill
            cell.font = Font(
                bold=True
            )
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True
            )
            cell.border = thin_border

        # Body formatting
        for row in worksheet.iter_rows(
            min_row=2,
            max_row=worksheet.max_row,
            min_col=1,
            max_col=worksheet.max_column
        ):
            for cell in row:
                cell.border = thin_border

                if isinstance(
                    cell.value,
                    (int, float)
                ):
                    cell.alignment = Alignment(
                        horizontal="center"
                    )

        # Freeze header row
        worksheet.freeze_panes = "A2"

        # Add filter
        worksheet.auto_filter.ref = (
            worksheet.dimensions
        )

        # Adjust column widths
        for column_cells in worksheet.columns:

            column_letter = (
                column_cells[0]
                .column_letter
            )

            max_length = 0

            for cell in column_cells:
                if cell.value is not None:
                    max_length = max(
                        max_length,
                        len(str(cell.value))
                    )

            worksheet.column_dimensions[
                column_letter
            ].width = min(
                max(
                    max_length + 2,
                    12
                ),
                28
            )

    # --------------------------------------------------------
    # Number formatting
    # --------------------------------------------------------

    ranking_sheet = workbook[
        "Top 2 Percent Ranking"
    ]

    ranking_headers = {
        cell.value: cell.column
        for cell in ranking_sheet[1]
    }

    detection_rate_column = (
        ranking_headers.get(
            "Detection Rate (%)"
        )
    )

    if detection_rate_column is not None:
        for row_number in range(
            2,
            ranking_sheet.max_row + 1
        ):
            ranking_sheet.cell(
                row=row_number,
                column=detection_rate_column
            ).number_format = "0.00"

    threshold_column = (
        ranking_headers.get(
            "Score Threshold"
        )
    )

    if threshold_column is not None:
        for row_number in range(
            2,
            ranking_sheet.max_row + 1
        ):
            ranking_sheet.cell(
                row=row_number,
                column=threshold_column
            ).number_format = "0.000000"

print("\nExcel workbook saved successfully.")
print("File:", excel_output_path.resolve())

In [ ]:
from pathlib import Path
import gc

import numpy as np
import pandas as pd
from openpyxl.styles import Font


# ============================================================
# 1. Paths
# ============================================================

BASE_DIR = Path(
    os.environ.get("GIS_ANOMALY_WORKSPACE_ROOT", ".")
    r"\Outlier Paper\Code"
)

# Clean proposed-method parameters
CLEAN_RESULTS_DIR = (
    BASE_DIR
    / "03_Proposed_Results_Seed_42"
)

# Parent folder containing all injection-ratio folders
INJECTED_RESULTS_ROOT = (
    BASE_DIR
    / "03_Injected_Proposed_Results_Seed_42"
)

# Select the injection experiment to process
INJECTION_FOLDER_NAME = (
    "Injected_Datasets_with_0.5%_Injection_Rate"
)

INJECTION_RESULTS_DIR = (
    INJECTED_RESULTS_ROOT
    / INJECTION_FOLDER_NAME
)

INJECTED_INDEX_PATH = (
    INJECTION_RESULTS_DIR
    / "inject_indices.npy"
)

FINAL_PARAMETER_PATH = (
    CLEAN_RESULTS_DIR
    / "full_pure_final_normalization_params.npz"
)

OUTPUT_CSV = (
    INJECTION_RESULTS_DIR
    / "Injected_Proposed_Detection_Rates_Raw_Final_Scores.csv"
)

OUTPUT_EXCEL = (
    INJECTION_RESULTS_DIR
    / "Injected_Proposed_Detection_Rates_Raw_Final_Scores.xlsx"
)


# ============================================================
# 2. Experimental settings
# ============================================================

SCENARIOS = [
    "extreme",
    "inconsistent",
    "noise_mild",
    "noise_moderate",
    "noise_strong",
]

# Exactly the same selection budgets used for the baselines
SELECTION_BUDGETS = [
    0.0010,   # top 0.10%
    0.0025,   # top 0.25%
    0.0050,   # top 0.50%
    0.0100,   # top 1.00%
    0.0200,   # top 2.00%
]

# Injection ratio represented by the selected folder
EXPECTED_INJECTION_RATIO = 0.0050


# ============================================================
# 3. File and array validation
# ============================================================

def require_file(path: Path) -> None:
    """Raise an informative error when a required file is missing."""

    if not path.exists():
        raise FileNotFoundError(
            f"Required file was not found:\n{path}"
        )


def load_1d_array(
    path: Path,
    expected_length: int | None = None,
    array_name: str | None = None,
) -> np.ndarray:
    """Load and validate a numeric one-dimensional score array."""

    require_file(path)

    array = np.load(
        path,
        mmap_mode="r",
    )

    array = np.asarray(
        array
    ).reshape(-1)

    name = (
        array_name
        if array_name is not None
        else path.name
    )

    if expected_length is not None:
        if array.size != expected_length:
            raise ValueError(
                f"{name} contains {array.size:,} values, "
                f"but {expected_length:,} were expected."
            )

    if not np.issubdtype(
        array.dtype,
        np.number,
    ):
        raise TypeError(
            f"{name} is not numeric."
        )

    if not np.isfinite(array).all():
        bad_count = int(
            (~np.isfinite(array)).sum()
        )

        raise ValueError(
            f"{name} contains {bad_count:,} "
            "NaN or infinite values."
        )

    return array


def load_reliability(
    path: Path,
    expected_length: int,
    array_name: str,
) -> np.ndarray:
    """Load the reliability array from a diagnostics NPZ file."""

    require_file(path)

    with np.load(path) as diagnostics:

        if "reliability" not in diagnostics.files:
            raise KeyError(
                f"'reliability' is missing from:\n{path}"
            )

        reliability = np.asarray(
            diagnostics["reliability"]
        ).reshape(-1)

    if reliability.size != expected_length:
        raise ValueError(
            f"{array_name} contains "
            f"{reliability.size:,} values, "
            f"but {expected_length:,} were expected."
        )

    if not np.isfinite(reliability).all():
        raise ValueError(
            f"{array_name} contains NaN or infinite values."
        )

    return reliability.astype(
        np.float32,
        copy=False,
    )


# ============================================================
# 4. Original clipped-normalization function
# ============================================================

def normalize_with_clean_bounds(
    scores: np.ndarray,
    lower_bound: float,
    upper_bound: float,
) -> np.ndarray:
    """
    Reproduce the percentile clipping and normalization used
    in the original proposed-method notebook.
    """

    if upper_bound <= lower_bound:
        raise ValueError(
            "The upper normalization bound must exceed "
            "the lower normalization bound."
        )

    normalized = (
        np.clip(
            scores,
            lower_bound,
            upper_bound,
        )
        - lower_bound
    ) / (
        upper_bound
        - lower_bound
    )

    return np.clip(
        normalized,
        0.0,
        1.0,
    ).astype(
        np.float32,
        copy=False,
    )


# ============================================================
# 5. Exact top-k ranking
# ============================================================

def create_top_ranking(
    scores: np.ndarray,
    maximum_k: int,
) -> np.ndarray:
    """
    Rank the highest maximum_k samples.

    Larger scores indicate greater anomalousness.

    Exact score ties are resolved using the sample index,
    with the lower index ranked first.
    """

    scores = np.asarray(
        scores
    ).reshape(-1)

    n_samples = scores.size

    if not 1 <= maximum_k <= n_samples:
        raise ValueError(
            f"maximum_k must be between 1 and {n_samples:,}."
        )

    boundary_position = (
        n_samples
        - maximum_k
    )

    boundary_score = np.partition(
        scores,
        boundary_position,
    )[boundary_position]

    indices_above_boundary = np.flatnonzero(
        scores > boundary_score
    )

    indices_at_boundary = np.flatnonzero(
        scores == boundary_score
    )

    number_needed_from_boundary = (
        maximum_k
        - indices_above_boundary.size
    )

    if number_needed_from_boundary < 0:
        raise RuntimeError(
            "Unexpected top-k boundary calculation."
        )

    selected_candidates = np.concatenate(
        [
            indices_above_boundary,
            indices_at_boundary[
                :number_needed_from_boundary
            ],
        ]
    )

    if selected_candidates.size != maximum_k:
        raise RuntimeError(
            "Failed to select the exact requested number "
            "of samples."
        )

    # Descending score; ascending index for exact ties
    ordering = np.lexsort(
        (
            selected_candidates,
            -scores[selected_candidates],
        )
    )

    return selected_candidates[
        ordering
    ]


# ============================================================
# 6. Evaluate all budgets from one ranking
# ============================================================

def evaluate_detection_rates(
    scores: np.ndarray,
    injected_indices: np.ndarray,
    budgets: list[float],
) -> list[dict]:
    """Calculate injected-anomaly detection rate at each budget."""

    scores = np.asarray(
        scores
    ).reshape(-1)

    n_samples = scores.size

    maximum_selected_count = int(
        np.ceil(
            max(budgets)
            * n_samples
        )
    )

    ranking = create_top_ranking(
        scores=scores,
        maximum_k=maximum_selected_count,
    )

    results = []

    for budget in budgets:

        selected_count = int(
            np.ceil(
                budget
                * n_samples
            )
        )

        selected_indices = ranking[
            :selected_count
        ]

        detected_count = int(
            np.isin(
                injected_indices,
                selected_indices,
                assume_unique=True,
            ).sum()
        )

        missed_count = int(
            injected_indices.size
            - detected_count
        )

        detection_rate = (
            detected_count
            / injected_indices.size
            * 100.0
        )

        miss_rate = (
            missed_count
            / injected_indices.size
            * 100.0
        )

        selection_threshold = float(
            scores[
                selected_indices[-1]
            ]
        )

        results.append(
            {
                "Budget_Fraction": budget,
                "Budget_Percent": budget * 100.0,
                "Selected_Count": selected_count,
                "Total_Injected": int(
                    injected_indices.size
                ),
                "Detected_Injected": detected_count,
                "Missed_Injected": missed_count,
                "Detection_Rate_Percent": detection_rate,
                "Miss_Rate_Percent": miss_rate,
                "Raw_Final_Score_Threshold": (
                    selection_threshold
                ),
            }
        )

    return results


# ============================================================
# 7. Load injected indices
# ============================================================

require_file(INJECTED_INDEX_PATH)
require_file(FINAL_PARAMETER_PATH)

inject_idx = np.load(
    INJECTED_INDEX_PATH
).astype(
    np.int64,
    copy=False,
)

inject_idx = np.asarray(
    inject_idx
).reshape(-1)

if inject_idx.size == 0:
    raise ValueError(
        "inject_indices.npy is empty."
    )

if np.any(inject_idx < 0):
    raise ValueError(
        "inject_indices.npy contains negative indices."
    )

if np.unique(inject_idx).size != inject_idx.size:
    raise ValueError(
        "inject_indices.npy contains duplicate indices."
    )

print("Injection folder:")
print(INJECTION_RESULTS_DIR)

print(
    "\nNumber of injected samples:",
    f"{inject_idx.size:,}"
)


# ============================================================
# 8. Load clean Full/Pure parameters
# ============================================================

required_parameter_keys = [
    "pure_base_weight",
    "pure_spatial_weight",
    "pure_spatial_p1",
    "pure_spatial_p99",
    "full_final_p1",
    "full_final_p99",
    "pure_final_p1",
    "pure_final_p99",
]

with np.load(
    FINAL_PARAMETER_PATH
) as parameter_file:

    missing_keys = [
        key
        for key in required_parameter_keys
        if key not in parameter_file.files
    ]

    if missing_keys:
        raise KeyError(
            "The following parameters are missing from "
            "full_pure_final_normalization_params.npz:\n"
            + ", ".join(missing_keys)
        )

    parameters = {
        key: float(
            parameter_file[key]
        )
        for key in required_parameter_keys
    }

pure_base_weight = parameters[
    "pure_base_weight"
]

pure_spatial_weight = parameters[
    "pure_spatial_weight"
]

pure_spatial_p1 = parameters[
    "pure_spatial_p1"
]

pure_spatial_p99 = parameters[
    "pure_spatial_p99"
]

full_final_p1 = parameters[
    "full_final_p1"
]

full_final_p99 = parameters[
    "full_final_p99"
]

pure_final_p1 = parameters[
    "pure_final_p1"
]

pure_final_p99 = parameters[
    "pure_final_p99"
]

print("\nLoaded Pure weights:")
print("AE weight     :", pure_base_weight)
print("Spatial weight:", pure_spatial_weight)


# ============================================================
# 9. Reconstruct and evaluate every scenario
# ============================================================

all_results = []
validation_results = []
score_summaries = []

expected_n_samples = None

for scenario in SCENARIOS:

    scenario_dir = (
        INJECTION_RESULTS_DIR
        / scenario
    )

    if not scenario_dir.exists():
        raise FileNotFoundError(
            f"Scenario folder was not found:\n{scenario_dir}"
        )

    print("\n" + "=" * 78)
    print("Scenario:", scenario)
    print("=" * 78)

    # ========================================================
    # FULL
    # ========================================================

    full_spatial_path = (
        scenario_dir
        / "full_spatial_score_normalized.npy"
    )

    full_reliability_path = (
        scenario_dir
        / "full_reliability_diagnostics.npz"
    )

    saved_full_normalized_path = (
        scenario_dir
        / "full_final_anomaly_score.npy"
    )

    full_spatial = load_1d_array(
        full_spatial_path,
        expected_length=expected_n_samples,
        array_name=(
            f"{scenario} Full spatial score"
        ),
    )

    if expected_n_samples is None:
        expected_n_samples = full_spatial.size

        if inject_idx.max() >= expected_n_samples:
            raise IndexError(
                "At least one injected index exceeds "
                "the score-array length."
            )

        expected_injected_count = int(
            np.floor(
                expected_n_samples
                * EXPECTED_INJECTION_RATIO
            )
        )

        if inject_idx.size != expected_injected_count:
            raise ValueError(
                "The injection folder does not match "
                "EXPECTED_INJECTION_RATIO.\n"
                f"Expected injected samples: "
                f"{expected_injected_count:,}\n"
                f"Found injected samples: "
                f"{inject_idx.size:,}"
            )

        print(
            "Number of samples:",
            f"{expected_n_samples:,}"
        )

        print(
            "Actual injection ratio:",
            f"{inject_idx.size / expected_n_samples * 100:.6f}%"
        )

    full_reliability = load_reliability(
        full_reliability_path,
        expected_length=expected_n_samples,
        array_name=(
            f"{scenario} Full reliability"
        ),
    )

    # Exact reconstruction of the notebook's
    # A_final_full_injected_raw
    full_final_raw = (
        full_spatial
        * full_reliability
    ).astype(
        np.float32,
        copy=False,
    )

    full_raw_output_path = (
        scenario_dir
        / "full_final_anomaly_score_raw.npy"
    )

    np.save(
        full_raw_output_path,
        full_final_raw,
    )

    # --------------------------------------------------------
    # Validate Full reconstruction
    # --------------------------------------------------------

    reconstructed_full_normalized = (
        normalize_with_clean_bounds(
            full_final_raw,
            full_final_p1,
            full_final_p99,
        )
    )

    saved_full_normalized = load_1d_array(
        saved_full_normalized_path,
        expected_length=expected_n_samples,
        array_name=(
            f"{scenario} saved Full final score"
        ),
    )

    full_max_difference = float(
        np.max(
            np.abs(
                reconstructed_full_normalized
                - saved_full_normalized
            )
        )
    )

    full_validation_passed = bool(
        np.allclose(
            reconstructed_full_normalized,
            saved_full_normalized,
            rtol=1e-6,
            atol=1e-6,
        )
    )

    validation_results.append(
        {
            "Scenario": scenario,
            "Method": "Full",
            "Validation_Passed": (
                full_validation_passed
            ),
            "Maximum_Absolute_Difference": (
                full_max_difference
            ),
            "Reconstructed_Raw_File": str(
                full_raw_output_path
            ),
        }
    )

    if not full_validation_passed:
        raise ValueError(
            f"Full reconstruction validation failed "
            f"for scenario '{scenario}'.\n"
            "The files may originate from different runs."
        )

    full_results = evaluate_detection_rates(
        scores=full_final_raw,
        injected_indices=inject_idx,
        budgets=SELECTION_BUDGETS,
    )

    for result in full_results:
        result["Scenario"] = scenario
        result["Method"] = "Full"
        all_results.append(result)

        print(
            f"Full | Top "
            f"{result['Budget_Percent']:.2f}% | "
            f"Detected "
            f"{result['Detected_Injected']:,}/"
            f"{result['Total_Injected']:,} | "
            f"DR="
            f"{result['Detection_Rate_Percent']:.4f}%"
        )

    score_summaries.append(
        {
            "Scenario": scenario,
            "Method": "Full",
            "Minimum_Raw_Final_Score": float(
                full_final_raw.min()
            ),
            "Maximum_Raw_Final_Score": float(
                full_final_raw.max()
            ),
            "Mean_Raw_Final_Score": float(
                full_final_raw.mean()
            ),
            "Raw_Final_Score_File": str(
                full_raw_output_path
            ),
        }
    )

    del full_spatial
    del full_reliability
    del reconstructed_full_normalized
    del saved_full_normalized
    del full_final_raw

    gc.collect()

    # ========================================================
    # PURE
    # ========================================================

    ae_score_path = (
        scenario_dir
        / "ae_score_normalized.npy"
    )

    spatial_score_path = (
        scenario_dir
        / "spatial_score_normalized.npy"
    )

    pure_reliability_path = (
        scenario_dir
        / "pure_reliability_diagnostics.npz"
    )

    saved_pure_normalized_path = (
        scenario_dir
        / "pure_final_anomaly_score.npy"
    )

    ae_score_normalized = load_1d_array(
        ae_score_path,
        expected_length=expected_n_samples,
        array_name=(
            f"{scenario} normalized AE score"
        ),
    )

    spatial_score_normalized = load_1d_array(
        spatial_score_path,
        expected_length=expected_n_samples,
        array_name=(
            f"{scenario} normalized spatial score"
        ),
    )

    # Reconstruct A_spatial_pure_injected
    pure_spatial_before_normalization = (
        pure_base_weight
        * ae_score_normalized
        +
        pure_spatial_weight
        * spatial_score_normalized
    ).astype(
        np.float32,
        copy=False,
    )

    # Reconstruct A_spatial_pure_injected_norm
    pure_spatial_normalized = (
        normalize_with_clean_bounds(
            pure_spatial_before_normalization,
            pure_spatial_p1,
            pure_spatial_p99,
        )
    )

    pure_reliability = load_reliability(
        pure_reliability_path,
        expected_length=expected_n_samples,
        array_name=(
            f"{scenario} Pure reliability"
        ),
    )

    # Exact reconstruction of the notebook's
    # A_final_pure_injected_raw
    pure_final_raw = (
        pure_spatial_normalized
        * pure_reliability
    ).astype(
        np.float32,
        copy=False,
    )

    pure_raw_output_path = (
        scenario_dir
        / "pure_final_anomaly_score_raw.npy"
    )

    np.save(
        pure_raw_output_path,
        pure_final_raw,
    )

    # --------------------------------------------------------
    # Validate Pure reconstruction
    # --------------------------------------------------------

    reconstructed_pure_normalized = (
        normalize_with_clean_bounds(
            pure_final_raw,
            pure_final_p1,
            pure_final_p99,
        )
    )

    saved_pure_normalized = load_1d_array(
        saved_pure_normalized_path,
        expected_length=expected_n_samples,
        array_name=(
            f"{scenario} saved Pure final score"
        ),
    )

    pure_max_difference = float(
        np.max(
            np.abs(
                reconstructed_pure_normalized
                - saved_pure_normalized
            )
        )
    )

    pure_validation_passed = bool(
        np.allclose(
            reconstructed_pure_normalized,
            saved_pure_normalized,
            rtol=1e-6,
            atol=1e-6,
        )
    )

    validation_results.append(
        {
            "Scenario": scenario,
            "Method": "Pure",
            "Validation_Passed": (
                pure_validation_passed
            ),
            "Maximum_Absolute_Difference": (
                pure_max_difference
            ),
            "Reconstructed_Raw_File": str(
                pure_raw_output_path
            ),
        }
    )

    if not pure_validation_passed:
        raise ValueError(
            f"Pure reconstruction validation failed "
            f"for scenario '{scenario}'.\n"
            "The files may originate from different runs."
        )

    pure_results = evaluate_detection_rates(
        scores=pure_final_raw,
        injected_indices=inject_idx,
        budgets=SELECTION_BUDGETS,
    )

    for result in pure_results:
        result["Scenario"] = scenario
        result["Method"] = "Pure"
        all_results.append(result)

        print(
            f"Pure | Top "
            f"{result['Budget_Percent']:.2f}% | "
            f"Detected "
            f"{result['Detected_Injected']:,}/"
            f"{result['Total_Injected']:,} | "
            f"DR="
            f"{result['Detection_Rate_Percent']:.4f}%"
        )

    score_summaries.append(
        {
            "Scenario": scenario,
            "Method": "Pure",
            "Minimum_Raw_Final_Score": float(
                pure_final_raw.min()
            ),
            "Maximum_Raw_Final_Score": float(
                pure_final_raw.max()
            ),
            "Mean_Raw_Final_Score": float(
                pure_final_raw.mean()
            ),
            "Raw_Final_Score_File": str(
                pure_raw_output_path
            ),
        }
    )

    del ae_score_normalized
    del spatial_score_normalized
    del pure_spatial_before_normalization
    del pure_spatial_normalized
    del pure_reliability
    del reconstructed_pure_normalized
    del saved_pure_normalized
    del pure_final_raw

    gc.collect()

    print(
        f"{scenario}: reconstruction and validation passed."
    )


# ============================================================
# 10. Create result tables
# ============================================================

results_df = pd.DataFrame(
    all_results
)

results_df = results_df[
    [
        "Scenario",
        "Method",
        "Budget_Fraction",
        "Budget_Percent",
        "Selected_Count",
        "Total_Injected",
        "Detected_Injected",
        "Missed_Injected",
        "Detection_Rate_Percent",
        "Miss_Rate_Percent",
        "Raw_Final_Score_Threshold",
    ]
]

results_df = results_df.sort_values(
    by=[
        "Scenario",
        "Budget_Fraction",
        "Method",
    ],
    ignore_index=True,
)

validation_df = pd.DataFrame(
    validation_results
)

score_summary_df = pd.DataFrame(
    score_summaries
)


# Detection-rate matrix
detection_summary_df = results_df.pivot_table(
    index=[
        "Scenario",
        "Method",
    ],
    columns="Budget_Percent",
    values="Detection_Rate_Percent",
    aggfunc="first",
).reset_index()

detection_summary_df.columns.name = None


# Detected-count matrix
detected_count_summary_df = results_df.pivot_table(
    index=[
        "Scenario",
        "Method",
    ],
    columns="Budget_Percent",
    values="Detected_Injected",
    aggfunc="first",
).reset_index()

detected_count_summary_df.columns.name = None


# Average detection rate across scenarios
average_detection_df = (
    results_df
    .groupby(
        [
            "Method",
            "Budget_Fraction",
            "Budget_Percent",
        ],
        as_index=False,
    )
    .agg(
        Mean_Detection_Rate_Percent=(
            "Detection_Rate_Percent",
            "mean",
        ),
        Standard_Deviation=(
            "Detection_Rate_Percent",
            "std",
        ),
        Minimum_Detection_Rate_Percent=(
            "Detection_Rate_Percent",
            "min",
        ),
        Maximum_Detection_Rate_Percent=(
            "Detection_Rate_Percent",
            "max",
        ),
    )
)


# Experimental settings
settings_df = pd.DataFrame(
    {
        "Setting": [
            "Injection results folder",
            "Expected injection ratio",
            "Actual injected samples",
            "Number of samples",
            "Selection budgets",
            "Full ranking score",
            "Pure ranking score",
            "Selection procedure",
            "Evaluation metric",
        ],
        "Value": [
            str(INJECTION_RESULTS_DIR),
            EXPECTED_INJECTION_RATIO,
            inject_idx.size,
            expected_n_samples,
            "0.10%, 0.25%, 0.50%, 1.00%, 2.00%",
            (
                "Full spatial normalized score "
                "multiplied by Full reliability"
            ),
            (
                "Normalized Pure spatial fusion "
                "multiplied by Pure reliability"
            ),
            "Exact top-k from pre-final-normalization score",
            "Injected-anomaly detection rate",
        ],
    }
)


# ============================================================
# 11. Save CSV
# ============================================================

results_df.to_csv(
    OUTPUT_CSV,
    index=False,
)

print("\nCSV saved to:")
print(OUTPUT_CSV)


# ============================================================
# 12. Save Excel workbook
# ============================================================

with pd.ExcelWriter(
    OUTPUT_EXCEL,
    engine="openpyxl",
) as writer:

    results_df.to_excel(
        writer,
        sheet_name="Detailed Results",
        index=False,
    )

    detection_summary_df.to_excel(
        writer,
        sheet_name="Detection Rate Summary",
        index=False,
    )

    detected_count_summary_df.to_excel(
        writer,
        sheet_name="Detected Count Summary",
        index=False,
    )

    average_detection_df.to_excel(
        writer,
        sheet_name="Average Across Scenarios",
        index=False,
    )

    validation_df.to_excel(
        writer,
        sheet_name="Reconstruction Validation",
        index=False,
    )

    score_summary_df.to_excel(
        writer,
        sheet_name="Raw Final Score Summary",
        index=False,
    )

    settings_df.to_excel(
        writer,
        sheet_name="Experimental Settings",
        index=False,
    )

    workbook = writer.book

    for worksheet in workbook.worksheets:

        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions

        for header_cell in worksheet[1]:
            header_cell.font = Font(
                bold=True
            )

        for column_cells in worksheet.columns:

            column_letter = (
                column_cells[0].column_letter
            )

            maximum_length = max(
                len(
                    str(cell.value)
                )
                if cell.value is not None
                else 0
                for cell in column_cells
            )

            worksheet.column_dimensions[
                column_letter
            ].width = min(
                maximum_length + 2,
                50,
            )


print("\nExcel workbook saved to:")
print(OUTPUT_EXCEL)

print("\nAll Full and Pure raw final scores were")
print("reconstructed, validated, and evaluated.")